In [ ]:
# Sesión 3 del módulo 8, clase 26/08
# Que los estudiantes entiendan, diseñen y apliquen redes neuronales artificiales en Python (usando Keras/TensorFlow) para resolver problemas reales de:

# Regresión (predecir valores continuos como precios o temperaturas).

# Clasificación (predecir categorías como especies de flores o si un pasajero del Titanic sobrevivió).

# Objetivos específicos implícitos en la guía

# Familiarizarse con el entorno Python para redes neuronales

# Uso de librerías como Keras y PyTorch.

# Diferenciar sus fortalezas (Keras = simple, PyTorch = flexible).

# Comprender qué es un tensor

# Escalar, vector, matriz, tensores 3D/4D.

# Cómo se usan para representar datos en RN (ej. imágenes, secuencias).

# Diseñar arquitecturas de redes neuronales

# Para regresión: salida lineal, pérdida MSE.

# Para clasificación: salida Sigmoid o Softmax, pérdidas de crossentropy.

# Aplicar funciones de activación

# ReLU, Sigmoid, Tanh, Softmax.

# Comprender cuándo usar cada una.

# Aprender el ciclo completo de un proyecto con RN

# Cargar/preparar datos.

# Definir modelo.

# Entrenar (fit).

# Evaluar y visualizar resultados.

# Hacer predicciones nuevas.


In [ ]:

# 1) Preparación básica (importaciones, semilla, versiones)
# @title Preparación: imports y configuración
import numpy as np
import tensorflow as tf
from tensorflow import keras
from keras import layers
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
# Importa TensorFlow, la librería de Google para Deep Learning.

# keras: interfaz de alto nivel dentro de TensorFlow, más sencilla para definir modelos.
# layers: módulo de Keras que contiene los bloques básicos de una red (Dense, Conv2D, Dropout, etc.)
# Opcional: fijar semillas para reproducibilidad (aun así, GPUs pueden variar un poco)
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
# Establece una semilla aleatoria fija.
# Esto garantiza que los números aleatorios (como inicialización de pesos, división del dataset, etc.) sean reproducibles.

print("TensorFlow:", tf.__version__)

TensorFlow: 2.16.2


In [2]:
# 2) Mini-intro a tensores (rápido y visual)
# @title Tensores: qué son y cómo se ven
# Un tensor es como un array de N dimensiones (0D escalar, 1D vector, 2D matriz, 3D+ ...)

# Escalar (0D)
escalar = tf.constant(3.14)
print("Escalar:", escalar, "shape:", escalar.shape)
# tf.constant(3.14) crea un tensor escalar (un solo número).
# Su forma (shape) es vacía: () porque no tiene dimensiones.
# Output: Escalar: tf.Tensor(3.14, shape=(), dtype=float32).

# Vector (1D)
vector = tf.constant([1, 2, 3], dtype=tf.float32)
print("Vector:", vector, "shape:", vector.shape)
# Crea un vector de 3 elementos (dimensión 1).
# Forma (shape) = (3,).
# Sirve para representar datos como una lista de características.

# Matriz (2D)
matriz = tf.constant([[1., 2.], [3., 4.]], dtype=tf.float32)
print("Matriz:\n", matriz.numpy(), "shape:", matriz.shape)
# Crea una matriz 2×2.
# Forma (shape) = (2,2).
# .numpy() lo convierte a formato NumPy para imprimirlo como tabla.
# Esto sería como una tabla de datos o una imagen en blanco y negro muy pequeña (2x2 pixeles).

# Tensor 3D (ej: 2 imágenes 2x2 con 1 canal)
tensor3d = tf.zeros((2, 2, 2, 1))
print("Tensor 4D (batch, alto, ancho, canales):", tensor3d.shape)
# Crea un tensor lleno de ceros con forma (2,2,2,1).
# Interprétalo como:
# 2 = número de imágenes en el batch.
# 2 = alto.
# 2 = ancho.
# 1 = canales (blanco y negro).
# Este es el formato típico en redes convolucionales para imágenes.

# 0D = escalar (un número).
# 1D = vector.
# 2D = matriz.
# 3D o más = datos más complejos (imágenes, videos, lotes de datos).

# Operaciones básicas
a = tf.constant([[1., 2.], [3., 4.]])
b = tf.constant([[5., 6.], [7., 8.]])
print("Suma:\n", (a + b).numpy())
print("Producto matriz:\n", tf.matmul(a, b).numpy())
# Suma elemento a elemento:

# [1 2; 3 4]+[5	6; 7 8]=[6 8; 10 12]

# Producto de matrices (tf.matmul):
# [1 2; 3 4]⋅[5 6; 7 8]=[19 22;43 50]

2025-08-29 18:18:02.072696: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M1
2025-08-29 18:18:02.072759: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 8.00 GB
2025-08-29 18:18:02.072805: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 2.67 GB
2025-08-29 18:18:02.073077: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-08-29 18:18:02.073226: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Escalar: tf.Tensor(3.14, shape=(), dtype=float32) shape: ()
Vector: tf.Tensor([1. 2. 3.], shape=(3,), dtype=float32) shape: (3,)
Matriz:
 [[1. 2.]
 [3. 4.]] shape: (2, 2)
Tensor 4D (batch, alto, ancho, canales): (2, 2, 2, 1)
Suma:
 [[ 6.  8.]
 [10. 12.]]
Producto matriz:
 [[19. 22.]
 [43. 50.]]


2) Vector
Vector: tf.Tensor([1. 2. 3.], shape=(3,), dtype=float32) shape: (3,)


tf.Tensor([1. 2. 3.], ...) → contiene 3 valores en una fila.

shape=(3,) → significa que tiene 1 dimensión con 3 elementos.

dtype=float32 → otra vez, números decimales de 32 bits.

Explicación simple: “Un vector es como una lista de 3 números”.

🔹 3) Matriz
Matriz:
 [[1. 2.]
 [3. 4.]] shape: (2, 2)


Representa una tabla de 2 filas × 2 columnas.

shape=(2,2) → confirma que es una matriz 2D con esas dimensiones.

Explicación simple: “Una matriz es como una tabla o cuadrícula de números”.

🔹 4) Tensor 4D
Tensor 4D (batch, alto, ancho, canales): (2, 2, 2, 1)


Es un tensor con 4 dimensiones:

2 → batch size (2 ejemplos en el conjunto).

2 → altura (2 pixeles).

2 → ancho (2 pixeles).

1 → canales (blanco y negro; si fueran colores sería 3).

Explicación simple: “Es como tener 2 imágenes muy pequeñas de 2×2 en blanco y negro”

In [ ]:
# 3) Regresión (valor continuo) – Datos sintéticos
# Objetivo: predecir un valor continuo (ej. “precio”).
# Arquitectura: red densa pequeña.
# Pérdida: MSE. Métrica: MAE.
# @title Regresión con datos sintéticos (make_regression)
from sklearn.datasets import make_regression

# 1) Datos sintéticos: 2 características, ruido moderado
X, y = make_regression(n_samples=1200, n_features=2, n_informative=2,
                       noise=12.0, random_state=SEED)

# 2) Train/Val/Test split
X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.30, random_state=SEED)
X_val, X_test, y_val, y_test   = train_test_split(X_tmp, y_tmp, test_size=0.50, random_state=SEED)

# 3) Escalado (muy importante en RN densas)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)

# 4) Modelo denso para regresión
model_reg = keras.Sequential([
    layers.Input(shape=(X_train_s.shape[1],)),
    layers.Dense(32, activation="relu"),
    layers.Dense(16, activation="relu"),
    layers.Dense(1, activation="linear")  # salida lineal en regresión
])

model_reg.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3),
                  loss="mse", metrics=["mae"])

# 5) Entrenamiento con EarlyStopping
early = keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True, monitor="val_loss")

hist = model_reg.fit(
    X_train_s, y_train,
    validation_data=(X_val_s, y_val),
    epochs=200,
    batch_size=32,
    callbacks=[early],
    verbose=0
)

print("Mejor val_loss:", np.min(hist.history["val_loss"]))

# 6) Curva de pérdida
plt.figure()
plt.plot(hist.history["loss"], label="train_loss")
plt.plot(hist.history["val_loss"], label="val_loss")
plt.xlabel("Época"); plt.ylabel("MSE"); plt.title("Entrenamiento (Regresión)")
plt.legend(); plt.show()

# 7) Evaluación en test
test_loss, test_mae = model_reg.evaluate(X_test_s, y_test, verbose=0)
print(f"Test MSE: {test_loss:.2f} | Test MAE: {test_mae:.2f}")

# 8) Predicción de ejemplo
ejemplo = np.array([[0.5, -1.0]])         # 2 características
ejemplo_s = scaler.transform(ejemplo)
pred = model_reg.predict(ejemplo_s, verbose=0)[0,0]
print("Predicción (valor continuo) para", ejemplo[0], "→", round(pred, 2))

El objetivo es predecir un número continuo (no una clase). Ejemplo: predecir el precio de una casa o la temperatura.
1) Datos sintéticos con make_regression
X, y = make_regression(n_samples=1200, n_features=2, n_informative=2,
                       noise=12.0, random_state=SEED)


Objetivo: crear un problema de regresión (predecir un valor continuo y a partir de 𝑋).

n_samples=1200: 1200 ejemplos.

n_features=2: 2 características por ejemplo (p.ej., “metros²” y “antigüedad”).

n_informative=2: las 2 características realmente aportan información.

noise=12.0: añade ruido (variabilidad) para que no sea trivial.

random_state=SEED: reproducibilidad.

Resultado: X tiene forma (1200, 2) y y es un vector (1200,) con valores reales.

MSE (Mean Squared Error): mide el error cuadrático medio → se usa como función de pérdida.

MAE (Mean Absolute Error): error absoluto medio → se usa como métrica para interpretar fácilmente.

🔹 1) Generación de datos sintéticos
from sklearn.datasets import make_regression

X, y = make_regression(n_samples=1200, n_features=2, n_informative=2,
                       noise=12.0, random_state=SEED)


Se crea un dataset artificial con 1200 muestras y 2 características (features).

n_informative=2: ambas características son relevantes.

noise=12.0: agrega ruido (variabilidad aleatoria) para que no sea un problema trivial.

X = matriz de entrada (1200 × 2).

y = vector de salida (1200 valores continuos).

Ejemplo: X = [metros², número de cuartos], y = precio.

🔹 2) División en entrenamiento, validación y prueba
X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.30, random_state=SEED)
X_val, X_test, y_val, y_test   = train_test_split(X_tmp, y_tmp, test_size=0.50, random_state=SEED)


70% entrenamiento → para que el modelo aprenda.

15% validación → para ajustar parámetros y evitar sobreajuste.

15% prueba → para medir rendimiento final.

Esta separación es fundamental para que el modelo generalice.

🔹 3) Escalado de datos
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)


Se estandarizan las características: media=0 y desviación estándar=1.

Muy importante en redes densas porque mejora la velocidad de entrenamiento y la convergencia.

fit_transform se hace solo con los datos de entrenamiento, y después se aplica transform a validación y prueba.

🔹 4) Definición del modelo
model_reg = keras.Sequential([
    layers.Input(shape=(X_train_s.shape[1],)),
    layers.Dense(32, activation="relu"),
    layers.Dense(16, activation="relu"),
    layers.Dense(1, activation="linear")  # salida lineal en regresión
])


Aquí se define una red neuronal densa pequeña:

Entrada: shape=(2,) porque hay 2 características de entrada.

Capa oculta 1: 32 neuronas con activación ReLU (introduce no linealidad).

Capa oculta 2: 16 neuronas con ReLU.

Capa de salida: 1 neurona con activación lineal, porque en regresión queremos predecir cualquier número real (positivo o negativo).

🔹 5) Compilación del modelo
model_reg.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3),
                  loss="mse", metrics=["mae"])


Optimizador: Adam (muy usado en Deep Learning, combina SGD + momentum + RMSProp).

Función de pérdida: "mse" porque es un problema de regresión.

Métrica: "mae" porque es más fácil de interpretar (ej. error medio = 10 unidades en el precio).

En resumen:

Generamos un dataset artificial de regresión.

Lo dividimos en train/val/test.

Escalamos los datos (importantísimo).

Creamos un modelo denso con Keras: entradas → capas ocultas con ReLU → salida lineal.

Compilamos el modelo con MSE como pérdida y MAE como métrica.

5) Entrenamiento con EarlyStopping
early = keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True, monitor="val_loss")


EarlyStopping = técnica para detener el entrenamiento antes de llegar a todas las épocas, cuando el modelo ya no mejora.

Parámetros:

patience=10: si durante 10 épocas seguidas no mejora la métrica monitoreada, se detiene el entrenamiento.

restore_best_weights=True: al final, restaura los pesos del modelo en el punto donde tuvo el mejor desempeño en validación.

monitor="val_loss": lo que vigila es la pérdida en el conjunto de validación.

 Beneficio: evita sobreajuste y ahorra tiempo de cómputo.

hist = model_reg.fit(
    X_train_s, y_train,
    validation_data=(X_val_s, y_val),
    epochs=200,
    batch_size=32,
    callbacks=[early],
    verbose=0
)


fit = entrena el modelo.

X_train_s, y_train: datos de entrenamiento.

validation_data: usa validación en cada época para monitorear.

epochs=200: máximo 200 iteraciones completas sobre los datos.

batch_size=32: entrena de a 32 muestras por lote.

callbacks=[early]: aplica EarlyStopping.

verbose=0: entrena en silencio (no imprime cada época).

 El resultado (hist) guarda todo el historial de pérdidas y métricas por época.

print("Mejor val_loss:", np.min(hist.history["val_loss"]))


 Muestra el mínimo error en validación (MSE) alcanzado durante el entrenamiento.
Sirve para saber hasta dónde llegó el modelo en su mejor punto.

🔹 6) Curva de pérdida
plt.figure()
plt.plot(hist.history["loss"], label="train_loss")
plt.plot(hist.history["val_loss"], label="val_loss")
plt.xlabel("Época"); plt.ylabel("MSE"); plt.title("Entrenamiento (Regresión)")
plt.legend(); plt.show()


Grafica la evolución del error cuadrático medio (MSE) en entrenamiento y validación.

Si las dos curvas bajan y se mantienen cercanas → el modelo aprende bien.

Si la de validación sube mientras la de entrenamiento sigue bajando → sobreajuste.

 Visualmente explica cómo aprende la red.

🔹 7) Evaluación en test
test_loss, test_mae = model_reg.evaluate(X_test_s, y_test, verbose=0)
print(f"Test MSE: {test_loss:.2f} | Test MAE: {test_mae:.2f}")


evaluate mide el desempeño en el conjunto de prueba (datos nunca vistos).

Devuelve:

test_loss: MSE en test.

test_mae: MAE en test (error promedio en las predicciones).

Ejemplo: Test MAE = 10.5 → en promedio, la predicción se equivoca en 10.5 unidades.

🔹 8) Predicción de ejemplo
ejemplo = np.array([[0.5, -1.0]])         # 2 características
ejemplo_s = scaler.transform(ejemplo)
pred = model_reg.predict(ejemplo_s, verbose=0)[0,0]
print("Predicción (valor continuo) para", ejemplo[0], "→", round(pred, 2))


Crea un ejemplo nuevo con 2 características (igual que el dataset).

Lo escala (scaler.transform) para que esté en la misma escala que el entrenamiento.

model_reg.predict devuelve la predicción → en este caso, un valor continuo (ej. precio).

[0,0] selecciona el primer valor del tensor de salida.

round(pred, 2) lo redondea a 2 decimales para mostrarlo bonito.

En resumen:

Se entrena la red con EarlyStopping para evitar sobreajuste.

Se visualizan las curvas de pérdida (train vs val).

Se evalúa en test para saber cómo generaliza.

Se hacen predicciones nuevas con datos de entrada desconocidos.

Explicación de la salida:
1) Curva de entrenamiento (la gráfica)

El eje x son las épocas (iteraciones completas sobre los datos).

El eje y es el MSE (Mean Squared Error).

La línea azul (train_loss) muestra el error en el conjunto de entrenamiento.

La línea naranja (val_loss) muestra el error en el conjunto de validación.

 Interpretación:

Al inicio (épocas 0–20) el error baja rápidamente → la red aprende los patrones principales.

Luego ambas curvas se estabilizan cerca de un valor bajo → el modelo converge sin sobreajuste (ambas curvas están casi juntas).

Eso significa que el modelo generaliza bien.

🔹 2) Mejor val_loss
Mejor val_loss: 161.18


El menor error cuadrático medio (MSE) alcanzado en validación fue 161.18.

Recuerda que MSE = promedio de los errores al cuadrado → un número más bajo = mejor ajuste.

Este valor es importante porque EarlyStopping guardó ese punto como el mejor modelo.

🔹 3) Evaluación en test
Test MSE: 153.30 | Test MAE: 10.04


Test MSE = 153.30 → error cuadrático medio en datos de prueba (muy similar al de validación, lo cual confirma que generaliza bien).

Test MAE = 10.04 → error absoluto medio.

Significa que en promedio, las predicciones se equivocan en 10 unidades respecto al valor real.

Es más fácil de interpretar que el MSE porque está en la misma escala que la variable objetivo.

 Esto nos dice que el modelo tiene un rendimiento consistente: ni sobreajustado ni con underfitting.

🔹 4) Predicción de ejemplo
Predicción (valor continuo) para [ 0.5 -1. ] → 20.32


El input fue [0.5, -1.0] (dos características).

Después de pasar por el modelo, la predicción fue 20.32.

Ese número es un valor continuo (porque es un problema de regresión).

Si el dataset fuera, por ejemplo, de precios, eso significaría “el modelo predice que para esas características, el precio será ≈ 20.32”.

In [ ]:
# 4) Clasificación multiclase – Iris (Setosa/Versicolor/Virginica)
# Objetivo: predecir una clase (3 categorías).
# Arquitectura: densa pequeña.
# Pérdida: SparseCategoricalCrossentropy (etiquetas enteras 0/1/2).
# Métricas: accuracy + reporte de clasificación.
# @title Clasificación multiclase con Iris
from sklearn.datasets import load_iris

# 1) Cargar Iris (150 muestras, 4 características, 3 clases)
iris = load_iris()
X, y = iris.data, iris.target

# 2) Train/Val/Test split
X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.30, random_state=SEED, stratify=y)
X_val, X_test, y_val, y_test   = train_test_split(X_tmp, y_tmp, test_size=0.50, random_state=SEED, stratify=y_tmp)

# 3) Escalado
scaler_clf = StandardScaler()
X_train_s = scaler_clf.fit_transform(X_train)
X_val_s   = scaler_clf.transform(X_val)
X_test_s  = scaler_clf.transform(X_test)

# 4) Modelo denso para clasificación multiclase
num_classes = len(np.unique(y))
model_clf = keras.Sequential([
    layers.Input(shape=(X_train_s.shape[1],)),
    layers.Dense(16, activation="relu"),
    layers.Dense(8, activation="relu"),
    layers.Dense(num_classes, activation="softmax")  # softmax -> probabilidades
])

model_clf.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])

# 5) Entrenamiento
early = keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True, monitor="val_loss")
hist2 = model_clf.fit(
    X_train_s, y_train,
    validation_data=(X_val_s, y_val),
    epochs=300,
    batch_size=16,
    callbacks=[early],
    verbose=0
)

print("Mejor val_accuracy:", np.max(hist2.history["val_accuracy"]))

# 6) Curvas de entrenamiento
plt.figure()
plt.plot(hist2.history["accuracy"], label="train_acc")
plt.plot(hist2.history["val_accuracy"], label="val_acc")
plt.xlabel("Época"); plt.ylabel("Accuracy"); plt.title("Entrenamiento (Clasificación Iris)")
plt.legend(); plt.show()

# 7) Evaluación y métricas detalladas
test_loss, test_acc = model_clf.evaluate(X_test_s, y_test, verbose=0)
print(f"Accuracy en test: {test_acc:.3f}")

y_pred_probs = model_clf.predict(X_test_s, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)

print("\nMatriz de confusión:\n", confusion_matrix(y_test, y_pred))
print("\nReporte de clasificación:\n", classification_report(y_test, y_pred, target_names=iris.target_names))

# 8) Predicción de ejemplo
ejemplo = X_test[:1]
ejemplo_s = scaler_clf.transform(ejemplo)
probs = model_clf.predict(ejemplo_s, verbose=0)[0]
print("Probabilidades por clase:", dict(zip(iris.target_names, np.round(probs,3))))
print("Clase predicha:", iris.target_names[np.argmax(probs)])

El objetivo es entrenar una red neuronal que clasifique flores en 3 especies:

Iris Setosa,

Iris Versicolor,

Iris Virginica.

Es un problema de clasificación multiclase.

🔹 1) Cargar el dataset Iris
from sklearn.datasets import load_iris

iris = load_iris()
X, y = iris.data, iris.target


iris.data (X) → matriz con 150 muestras × 4 características: largo y ancho de sépalo y pétalo.

iris.target (y) → vector con las clases:

0 = Setosa, 1 = Versicolor, 2 = Virginica.

🔹 2) División en entrenamiento, validación y prueba
X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.30, random_state=SEED, stratify=y)
X_val, X_test, y_val, y_test   = train_test_split(X_tmp, y_tmp, test_size=0.50, random_state=SEED, stratify=y_tmp)


Se divide el dataset en:

70% entrenamiento (X_train, y_train).

15% validación (X_val, y_val).

15% prueba (X_test, y_test).

stratify=y asegura que cada clase esté balanceada en los tres conjuntos.

🔹 3) Escalado de los datos
scaler_clf = StandardScaler()
X_train_s = scaler_clf.fit_transform(X_train)
X_val_s   = scaler_clf.transform(X_val)
X_test_s  = scaler_clf.transform(X_test)


Se normalizan las variables (media = 0, desviación = 1).

Esto es importante en redes densas porque facilita la convergencia del modelo.

Se ajusta (fit_transform) con entrenamiento y se aplica (transform) en validación y prueba → para no filtrar información.

🔹 4) Definición del modelo
num_classes = len(np.unique(y))
model_clf = keras.Sequential([
    layers.Input(shape=(X_train_s.shape[1],)),
    layers.Dense(16, activation="relu"),
    layers.Dense(8, activation="relu"),
    layers.Dense(num_classes, activation="softmax")  # softmax -> probabilidades
])


num_classes = 3 (porque Iris tiene 3 clases).

Arquitectura de la red:

Capa de entrada: 4 neuronas (porque el dataset tiene 4 características).

Capa oculta 1: 16 neuronas con activación ReLU.

Capa oculta 2: 8 neuronas con ReLU.

Capa de salida: 3 neuronas con softmax, que devuelve probabilidades para cada clase.

 Ejemplo de salida para una muestra:
[0.05, 0.10, 0.85] → el modelo predice 85% probabilidad de que sea Virginica.

🔹 5) Compilación del modelo
model_clf.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])


Optimizador: Adam con learning rate 0.001 → muy usado en DL.

Función de pérdida: sparse_categorical_crossentropy.

Se usa cuando las etiquetas están como enteros (0, 1, 2).

Si estuvieran en one-hot encoding, usaríamos categorical_crossentropy.

Métrica: "accuracy" → mide el porcentaje de clasificaciones correctas.

EarlyStopping: detiene el entrenamiento si el modelo deja de mejorar.

patience=15: espera hasta 15 épocas sin mejora antes de detenerse.

restore_best_weights=True: recupera los mejores pesos logrados durante el entrenamiento.

monitor="val_loss": se guía por la pérdida en validación.

Evita sobreajuste y ahorro de tiempo.

hist2 = model_clf.fit(
    X_train_s, y_train,
    validation_data=(X_val_s, y_val),
    epochs=300,
    batch_size=16,
    callbacks=[early],
    verbose=0
)


fit entrena el modelo.

X_train_s, y_train: datos de entrenamiento.

validation_data=(X_val_s, y_val): evalúa en validación cada época.

epochs=300: máximo de 300 épocas, aunque puede parar antes por EarlyStopping.

batch_size=16: procesa 16 muestras por vez.

verbose=0: no muestra el progreso (modo silencioso).

El resultado (hist2) guarda el historial de métricas.

print("Mejor val_accuracy:", np.max(hist2.history["val_accuracy"]))


Muestra el máximo accuracy en validación alcanzado durante el entrenamiento.

Ejemplo: Mejor val_accuracy: 0.97 → significa que la red clasificó correctamente el 97% en validación.

🔹 6) Curvas de entrenamiento
plt.figure()
plt.plot(hist2.history["accuracy"], label="train_acc")
plt.plot(hist2.history["val_accuracy"], label="val_acc")
plt.xlabel("Época"); plt.ylabel("Accuracy"); plt.title("Entrenamiento (Clasificación Iris)")
plt.legend(); plt.show()


Grafica la evolución de la precisión (accuracy) en entrenamiento y validación.

Si ambas curvas suben y se mantienen cercanas → buen aprendizaje.

Si train_acc sigue subiendo pero val_acc baja → posible sobreajuste.

🔹 7) Evaluación en test y métricas
test_loss, test_acc = model_clf.evaluate(X_test_s, y_test, verbose=0)
print(f"Accuracy en test: {test_acc:.3f}")


Evalúa el modelo en el conjunto de prueba (datos nunca vistos).

Muestra el accuracy → qué porcentaje de aciertos logró.

y_pred_probs = model_clf.predict(X_test_s, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)


y_pred_probs: devuelve las probabilidades por clase de cada muestra.

np.argmax(..., axis=1): selecciona la clase con mayor probabilidad (la predicción final).

print("\nMatriz de confusión:\n", confusion_matrix(y_test, y_pred))
print("\nReporte de clasificación:\n", classification_report(y_test, y_pred, target_names=iris.target_names))


Matriz de confusión: muestra cuántas muestras de cada clase fueron clasificadas correctamente o confundidas con otras.

Reporte de clasificación: incluye métricas por clase:

Precision (exactitud cuando predice una clase).

Recall (cobertura de los verdaderos positivos).

F1-score (promedio entre precisión y recall).

Muy útil para ver si el modelo confunde una clase más que otra.

🔹 8) Predicción de ejemplo
ejemplo = X_test[:1]
ejemplo_s = scaler_clf.transform(ejemplo)
probs = model_clf.predict(ejemplo_s, verbose=0)[0]
print("Probabilidades por clase:", dict(zip(iris.target_names, np.round(probs,3))))
print("Clase predicha:", iris.target_names[np.argmax(probs)]))


Toma una muestra de prueba (X_test[:1]).

La normaliza con el mismo scaler.

El modelo predice un vector de probabilidades por clase.
Ejemplo:

Probabilidades por clase: {'setosa': 0.01, 'versicolor': 0.05, 'virginica': 0.94}
Clase predicha: virginica


np.argmax(probs) selecciona la clase con probabilidad más alta.


 En resumen:

Se entrena la red con EarlyStopping.

Se ven las curvas de accuracy (train vs val).

Se evalúa en test → accuracy global.

Se usan métricas avanzadas: matriz de confusión + reporte de clasificación.

Se hacen predicciones individuales con probabilidades.

1) Gráfica de entrenamiento (accuracy)

Eje X = número de épocas (iteraciones de entrenamiento).

Eje Y = accuracy (precisión, proporción de aciertos).

Curva azul (train_acc): precisión en el conjunto de entrenamiento.

Curva naranja (val_acc): precisión en el conjunto de validación.

 Interpretación:

Al inicio (épocas bajas) la accuracy era baja, pero rápidamente subió.

La red alcanzó casi 100% en entrenamiento y alrededor de 86% en validación.

La brecha entre train y val sugiere un poco de sobreajuste, pero aún generaliza bien.

🔹 2) Mejor val_accuracy
Mejor val_accuracy: 0.8636


El mejor desempeño en validación fue de 86.36%.

Esto significa que en los datos de validación (no usados para entrenar), el modelo clasificó correctamente el 86% de las flores.

🔹 3) Accuracy en test
Accuracy en test: 1.000


En el conjunto de prueba (datos nunca vistos durante el entrenamiento), el modelo clasificó correctamente el 100% de los casos.

Esto es una excelente señal de que el modelo aprendió bien y generalizó.

🔹 4) Matriz de confusión
[[7 0 0]
 [0 8 0]
 [0 0 8]]


Filas = clases reales.

Columnas = clases predichas.

 Interpretación:

Para Setosa (7 muestras) → todas fueron clasificadas como Setosa.

Para Versicolor (8 muestras) → todas fueron clasificadas como Versicolor.

Para Virginica (8 muestras) → todas fueron clasificadas como Virginica.

 Ningún error → clasificación perfecta en test.

🔹 5) Reporte de clasificación
               precision    recall  f1-score   support

      setosa       1.00      1.00      1.00         7
  versicolor       1.00      1.00      1.00         8
   virginica       1.00      1.00      1.00         8


Precision = exactitud al predecir una clase.

Recall = proporción de verdaderos positivos detectados.

F1-score = balance entre precision y recall.

Support = cantidad de muestras de esa clase.

 Todas las métricas son 1.00 (100%) → perfecto en cada clase.

🔹 6) Predicción de ejemplo
Probabilidades por clase: {'setosa': 1.0, 'versicolor': 0.0, 'virginica': 0.0}
Clase predicha: setosa


El modelo asignó 100% de probabilidad a la clase Setosa para esa flor de prueba.

Por eso la predicción final es Setosa.

Resumen:

El modelo entrenó bien y generalizó correctamente.

Validación llegó a ~86%, pero en test dio 100%.

La matriz de confusión muestra que no hubo errores en test.

El reporte de clasificación confirma precisión perfecta en las 3 clases.

La predicción individual muestra cómo funciona el softmax → da probabilidades y elige la clase más alta.

In [ ]:
# 5) Demo rápida de funciones de activación (gráficas)
# @title Visualizar Sigmoide, Tanh y ReLU
x = np.linspace(-6, 6, 400)
sigmoid = 1/(1 + np.exp(-x))
tanh = np.tanh(x)
relu = np.maximum(0, x)

plt.figure(); plt.plot(x, sigmoid); plt.title("Sigmoide"); plt.grid(True); plt.show()
plt.figure(); plt.plot(x, tanh);    plt.title("Tanh");     plt.grid(True); plt.show()
plt.figure(); plt.plot(x, relu);    plt.title("ReLU");     plt.grid(True); plt.show()

1) Crear valores de entrada
x = np.linspace(-6, 6, 400)


Se crea un vector x con 400 valores equiespaciados entre -6 y 6.

Es el eje horizontal para graficar las funciones.
Piensa en él como la entrada de la neurona.

2) Definir funciones de activación
sigmoid = 1/(1 + np.exp(-x))

Devuelve valores entre 0 y 1.

Útil para probabilidades y clasificación binaria.

tanh = np.tanh(x)

Función tanh (tangente hiperbólica):
tanh(x)=(e^x-e^−x)/(e^x+e^−x)

Devuelve valores entre -1 y 1.

Centra los datos alrededor de 0, lo que ayuda al entrenamiento.

relu = np.maximum(0, x)

Función ReLU (Rectified Linear Unit):
f(x)=max(0,x)

Devuelve 0 si 𝑥<0 y 𝑥 si x≥0.

Es la más usada en capas ocultas porque es simple y evita problemas de gradientes pequeños.

3) Graficar cada función
plt.figure(); plt.plot(x, sigmoid); plt.title("Sigmoide"); plt.grid(True); plt.show()

Dibuja la curva sigmoide (forma de “S”).

plt.figure(); plt.plot(x, tanh);    plt.title("Tanh");     plt.grid(True); plt.show()


Dibuja la tangente hiperbólica (curva que va de -1 a 1).

plt.figure(); plt.plot(x, relu);    plt.title("ReLU");     plt.grid(True); plt.show()


Dibuja ReLU: una línea horizontal en 0 (para negativos) y una diagonal positiva (para positivos).

Resumen:

Sigmoide: aplasta valores entre 0 y 1 → útil para probabilidades.

Tanh: aplasta valores entre -1 y 1 → mejor para datos centrados.

ReLU: rápida, simple, ideal para capas ocultas porque deja pasar solo valores positivos.

Interpretación de los gráficos.
1) Sigmoide

Gráfico: curva en forma de “S” que va de 0 a 1.

Para valores muy negativos de x, la salida se acerca a 0.

Para valores muy positivos dex, la salida se acerca a 1.

En torno a x=0, la pendiente es más pronunciada → la red es más sensible a cambios.

Interpretación:

Convierte cualquier número real en un valor entre 0 y 1.

Se usa en la capa de salida de clasificación binaria, ya que se interpreta como una probabilidad.

Problema: en valores extremos la derivada es muy pequeña (gradiente casi cero) → desvanecimiento del gradiente.

🔹 2) Tanh (Tangente hiperbólica)

Gráfico: curva en forma de “S” pero centrada en el origen, con salida entre -1 y 1.

Para valores negativos grandes, se aplana en -1.

Para valores positivos grandes, se aplana en +1.

En el centro (x=0), la pendiente es máxima.

 Interpretación:

Similar a sigmoide, pero simétrica respecto al eje 0 → los valores están centrados entre negativo y positivo.

Esto ayuda al entrenamiento porque el modelo puede aprender valores positivos y negativos.

Problema: también sufre de gradientes muy pequeños en los extremos.

🔹 3) ReLU (Rectified Linear Unit)

 Gráfico:

Para valores negativos, la salida es 0 (línea horizontal).

Para valores positivos, la salida es igual a la entrada (f(x)=x).

 Interpretación:

Es la función más usada en capas ocultas.

Ventajas:

Computacionalmente simple.

Evita en gran parte el problema del desvanecimiento del gradiente.

Desventaja:

Para valores negativos, siempre es 0 → algunas neuronas pueden “morir” (no aprender nunca).

 Resumen:

Sigmoide: salida entre 0–1 → útil para probabilidades, pero puede saturarse.

Tanh: salida entre -1 y 1 → centrada en 0, mejor que sigmoide en capas ocultas.

ReLU: salida 0 para negativos y lineal para positivos → la más usada en capas ocultas por su simplicidad y eficiencia.

In [ ]:
# 6) (Opcional) Clasificación binaria mínima
# @title Binaria: ¿compra/no compra? (datos sintéticos)
from sklearn.datasets import make_classification

# 1) Datos binarios sintéticos
X, y = make_classification(n_samples=1000, n_features=6, n_informative=4,
                           n_redundant=0, random_state=SEED)

# 2) Split + escalado
X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.30, random_state=SEED, stratify=y)
X_val, X_test, y_val, y_test   = train_test_split(X_tmp, y_tmp, test_size=0.50, random_state=SEED, stratify=y_tmp)

sc = StandardScaler()
X_train_s = sc.fit_transform(X_train)
X_val_s   = sc.transform(X_val)
X_test_s  = sc.transform(X_test)

# 3) Modelo binario
model_bin = keras.Sequential([
    layers.Input(shape=(X_train_s.shape[1],)),
    layers.Dense(32, activation="relu"),
    layers.Dense(16, activation="relu"),
    layers.Dense(1, activation="sigmoid")  # una unidad + sigmoide en binaria
])
model_bin.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

early = keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True, monitor="val_loss")
hist3 = model_bin.fit(
    X_train_s, y_train,
    validation_data=(X_val_s, y_val),
    epochs=200, batch_size=32,
    callbacks=[early], verbose=0
)

# 4) Evaluación
loss, acc = model_bin.evaluate(X_test_s, y_test, verbose=0)
print(f"Accuracy (test): {acc:.3f}")

# 5) Predicción ejemplo
p = model_bin.predict(X_test_s[:5], verbose=0).reshape(-1)
print("Probabilidades predichas:", np.round(p, 3))
print("Clases predichas:", (p >= 0.5).astype(int))
print("Clases reales    :", y_test[:5])

1) Generar datos sintéticos
from sklearn.datasets import make_classification

X, y = make_classification(n_samples=1000, n_features=6, n_informative=4,
                           n_redundant=0, random_state=SEED)


make_classification crea un dataset artificial para clasificación.

n_samples=1000: 1000 ejemplos.

n_features=6: cada ejemplo tiene 6 características (edad, ingresos, etc., en un caso real).

n_informative=4: de esas 6, solo 4 aportan información útil.

n_redundant=0: no hay variables duplicadas.

y: las etiquetas binarias (0 o 1).

 Esto simula un dataset de clasificación binaria.

🔹 2) División y escalado
X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.30, random_state=SEED, stratify=y)
X_val, X_test, y_val, y_test   = train_test_split(X_tmp, y_tmp, test_size=0.50, random_state=SEED, stratify=y_tmp)

sc = StandardScaler()
X_train_s = sc.fit_transform(X_train)
X_val_s   = sc.transform(X_val)
X_test_s  = sc.transform(X_test)


Se dividen los datos en:

70% entrenamiento,

15% validación,

15% prueba.

stratify=y: asegura que la proporción de clases (0/1) sea similar en cada conjunto.

StandardScaler: estandariza las características (media=0, varianza=1) → importante para que la red aprenda mejor.

🔹 3) Definir el modelo binario
model_bin = keras.Sequential([
    layers.Input(shape=(X_train_s.shape[1],)),
    layers.Dense(32, activation="relu"),
    layers.Dense(16, activation="relu"),
    layers.Dense(1, activation="sigmoid")  # una unidad + sigmoide en binaria
])


Entrada: 6 características → shape=(6,).

Capas ocultas: 32 y 16 neuronas con activación ReLU.

Salida: 1 neurona con sigmoide, porque es un problema binario.

La sigmoide devuelve un valor entre 0 y 1 → interpretado como probabilidad de clase 1 (compra).

 Si la probabilidad ≥ 0.5 → se predice clase 1.
Si < 0.5 → se predice clase 0.

model_bin.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])


Optimizador: Adam.

Pérdida: binary_crossentropy → adecuada para clasificación binaria.

Métrica: accuracy.

🔹 4) Entrenamiento
early = keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True, monitor="val_loss")
hist3 = model_bin.fit(
    X_train_s, y_train,
    validation_data=(X_val_s, y_val),
    epochs=200, batch_size=32,
    callbacks=[early], verbose=0
)


Entrena hasta 200 épocas máximo, pero EarlyStopping detiene antes si no mejora la validación en 10 épocas seguidas.

batch_size=32: procesa 32 muestras por lote.

Usa validación para monitorear.

 Esto ahorra tiempo y evita sobreajuste.

🔹 5) Evaluación en test
loss, acc = model_bin.evaluate(X_test_s, y_test, verbose=0)
print(f"Accuracy (test): {acc:.3f}")


Evalúa el modelo en el conjunto de prueba.

Muestra la precisión global (ejemplo: Accuracy (test): 0.89 → 89% de aciertos).

🔹 6) Predicciones de ejemplo
p = model_bin.predict(X_test_s[:5], verbose=0).reshape(-1)
print("Probabilidades predichas:", np.round(p, 3))
print("Clases predichas:", (p >= 0.5).astype(int))
print("Clases reales    :", y_test[:5])


model_bin.predict devuelve probabilidades (ej: [0.8, 0.3, 0.65, ...]).

(p >= 0.5).astype(int) convierte probabilidades en clases binarias:

Si ≥ 0.5 → 1 (compra).

Si < 0.5 → 0 (no compra).

y_test[:5]: imprime las clases reales para comparar con las predichas.

Ejemplo de salida:

Probabilidades predichas: [0.812 0.276 0.654 0.901 0.102]
Clases predichas: [1 0 1 1 0]
Clases reales    : [1 0 1 1 0]


El modelo acertó en todas esas 5 muestras.

Resumen:

Generamos datos binarios simulados.

Dividimos y escalamos.

Creamos una red con salida sigmoide → probabilidades 0–1.

Usamos binary crossentropy como pérdida.

Evaluamos con accuracy.

Probamos ejemplos → el modelo devuelve probabilidades y se convierten en clases 0 o 1.

1) Exactitud en test
Accuracy (test): 0.887


El modelo tuvo un 88.7% de aciertos en el conjunto de prueba.

Es decir, casi 9 de cada 10 muestras fueron clasificadas correctamente.
 Esto indica que la red aprendió bien, aunque todavía puede cometer algunos errores.

🔹 2) Probabilidades predichas
Probabilidades predichas: [0.007 0.997 0.924 0.919 1.   ]


Estas son las salidas de la función sigmoide para las 5 primeras muestras del conjunto de prueba.

Cada valor indica la probabilidad de pertenecer a la clase 1 (compra):

0.007 ≈ 0% → muy baja probabilidad de compra.

0.997 = 99.7% → altísima probabilidad de compra.

0.924 = 92.4% → probabilidad alta de compra.

0.919 = 91.9% → probabilidad alta de compra.

1.000 = 100% → seguro que es compra.

🔹 3) Clases predichas
Clases predichas: [0 1 1 1 1]


Se aplica el umbral 0.5 a las probabilidades:

Si probabilidad ≥ 0.5 → Clase 1 (compra).

Si probabilidad < 0.5 → Clase 0 (no compra).

Interpretación:

La primera muestra fue predicha como 0 (no compra).

Las demás fueron predichas como 1 (compra).

🔹 4) Clases reales
Clases reales    : [0 1 1 1 1]


Estas son las etiquetas verdaderas de las mismas 5 muestras.

🔹 5) Comparación

Predicho: [0 1 1 1 1]

Real: [0 1 1 1 1]

 ¡Coinciden todas!
Eso significa que en estas 5 muestras específicas el modelo no se equivocó.

Resumen para tus estudiantes:

El modelo logra 88.7% de precisión global en test.

Devuelve probabilidades (ej. 0.997 ≈ “99.7% probabilidad de compra”).

Luego esas probabilidades se convierten en clases binarias (0/1) aplicando un umbral de 0.5.

En este ejemplo, las 5 predicciones coincidieron con las clases reales → el modelo clasificó perfectamente ese mini-lote.

In [ ]:
# Segundo ejemplo
import matplotlib.pyplot as plt

# 1. Preparar los datos
# x_train: Horas de estudio de 10 estudiantes
# y_train: Resultado del examen (0 = reprobado, 1 = aprobado)
# Aquí los datos son generados, pero en la vida real se cargarían de un archivo (ej. CSV)
x_train = np.array([0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0])
y_train = np.array([0, 0, 0, 0, 1, 1, 1, 1, 1, 1])

# 2. Construir la arquitectura de la red neuronal densa
# La red tendrá una capa de entrada, una capa oculta y una capa de salida.
# La capa de entrada se define implícitamente por el 'input_shape' de la primera capa oculta.
model = keras.Sequential([
    # Capa oculta con 4 neuronas y función de activación ReLU
    layers.Dense(4, activation='relu', input_shape=(1,)), # input_shape=(1,) indica una entrada de 1 valor (las horas de estudio)

    # Capa de salida con 1 neurona y función de activación Sigmoide para clasificación binaria
    layers.Dense(1, activation='sigmoid')
])

# 3. Compilar el modelo
# Se define el optimizador (cómo se ajustan los pesos), la función de pérdida (para medir el error) y la métrica de evaluación.
# 'binary_crossentropy' es la función de pérdida ideal para la clasificación binaria.
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

# Mostrar un resumen de la arquitectura de la red
model.summary()

# 4. Entrenar la red neuronal
# El modelo aprende de los datos de entrenamiento
print("\n--- Iniciando el entrenamiento ---")
history = model.fit(x_train, y_train, epochs=100, verbose=0) # epochs=100 significa que el modelo revisará los datos 100 veces

print("Entrenamiento finalizado.")

Model: "sequential_4": Este es el nombre del modelo.

Sequential es una de las APIs de Keras para construir modelos capa por capa. El número

4 indica que es la cuarta vez que se ha creado un modelo Sequential en esa sesión de ejecución, aunque a veces Keras asigna un número aleatorio a los modelos.

Layer (type): Muestra el nombre y el tipo de cada capa en la red. En este caso, ambas capas se llaman

dense_12 y dense_13, y son del tipo Dense. Una capa densa es un tipo de red en la que cada neurona de una capa está conectada a todas las neuronas de la capa siguiente.

Output Shape: Indica la forma de los datos que salen de cada capa.

(None, 4): La primera capa densa (dense_12) tiene 4 neuronas y su salida es un tensor con una forma de (None, 4). El None representa el tamaño del batch (grupo de datos) de entrada, que es flexible y puede variar. El número 4 corresponde a la cantidad de neuronas en esa capa, que es la cantidad de características que se pasan a la siguiente capa.

(None, 1): La segunda capa densa (dense_13), que es la capa de salida, tiene una sola neurona. Esto es lo típico para un problema de

clasificación binaria, donde la salida es un único valor (0 o 1). El

None también se refiere al tamaño del batch de entrada.

Param # (Número de Parámetros): Es el número total de parámetros entrenables en cada capa. Los parámetros son los pesos (

w) y los sesgos (b). El número se calcula de la siguiente manera:




Capa 1 (dense_12): La capa de entrada tiene un solo valor (x). La primera capa densa tiene 4 neuronas. La fórmula para los parámetros es: (número de entradas * número de neuronas) + número de neuronas (por el sesgo). En este caso:

(1 * 4) + 4 = 8.

Capa 2 (dense_13): Esta capa toma la salida de la capa anterior, que es un tensor con 4 valores. La fórmula es: (número de entradas * número de neuronas) + número de neuronas. En este caso: (4 * 1) + 1 = 5.

Total params: 13: La suma de los parámetros de todas las capas (8 + 5 = 13).


Trainable params: 13: Estos son los parámetros que el modelo ajusta durante el proceso de entrenamiento. En este modelo simple, todos los parámetros son entrenables.

Non-trainable params: 0: Son los parámetros que no se ajustan durante el entrenamiento. Esto puede suceder en modelos más complejos que utilizan técnicas como la transferencia de aprendizaje.

In [ ]:
# 5. Hacer una predicción y evaluar el modelo
# Predicción para un nuevo estudiante que estudió 3.2 horas
horas_estudio = np.array([3.2])
prediccion = model.predict(horas_estudio)

print(f"\n--- Resultado de la predicción ---")
print(f"Predicción para un estudiante que estudió 3.2 horas: {prediccion[0][0]:.4f}")

# El resultado es un valor entre 0 y 1. Si es > 0.5, se considera 'aprobado'.
if prediccion[0][0] > 0.5:
    print("El modelo predice que el estudiante aprobará el examen.")
else:
    print("El modelo predice que el estudiante reprobará el examen.")

# Visualizar el historial de entrenamiento (opcional, pero muy útil)
plt.plot(history.history['loss'])
plt.title('Pérdida (Loss) del Modelo')
plt.ylabel('Pérdida')
plt.xlabel('Época')
plt.show()

Este gráfico, titulado "Pérdida (Loss) del Modelo", es una representación visual del proceso de entrenamiento de la red neuronal.

Eje horizontal (Época): Representa el número de veces que el modelo ha visto y ha aprendido de todo el conjunto de datos de entrenamiento. En este caso, el entrenamiento duró 100 épocas.

Eje vertical (Pérdida): Muestra el valor de la función de pérdida del modelo. Esta función es una medida del error entre la predicción del modelo y el valor real. En otras palabras, te indica qué tan bien (o mal) se está desempeñando la red en cada época.

La curva: Muestra que la pérdida disminuye a medida que avanzan las épocas. Esto es lo que se espera en un entrenamiento exitoso, ya que el modelo ajusta sus pesos y sesgos para reducir el error en sus predicciones. La curva descendente indica que la red está aprendiendo correctamente y volviéndose más precisa con cada iteración.

Resultado de la Predicción
Después de entrenar el modelo, se utiliza para hacer una nueva predicción.

Predicción para un estudiante que estudió 3.2 horas: 0.3118: El modelo predijo un valor de 0.3118 para un estudiante que estudió 3.2 horas. En este tipo de problemas de clasificación binaria (donde solo hay dos posibles resultados, como "aprobar" o "reprobar"), la función de activación sigmoide en la capa de salida convierte el resultado en un valor entre 0 y 1.

El modelo predice que el estudiante reprobará el examen: El umbral común para interpretar la salida es 0.5.

Si el valor predicho es mayor a 0.5, se clasifica como 1 (en este caso, "aprobar").

Si el valor predicho es menor a 0.5, se clasifica como 0 (en este caso, "reprobar").

Dado que el valor de predicción es 0.3118, que es menor que 0.5, el modelo concluye que el estudiante reprobará el examen.

In [ ]:
# Ejemplo 3
# 1) Preparación e importaciones
# @title Preparación e importaciones
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report

# Reproducibilidad (puede variar levemente en GPU)
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)

In [ ]:
# 2) Cargar y explorar el dataset MNIST
# @title Carga de datos: MNIST (28x28, dígitos 0–9)
# MNIST viene integrado en Keras: 60k imágenes para train y 10k para test
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

print("Train:", x_train.shape, y_train.shape)
print("Test :", x_test.shape, y_test.shape)

# Visualizar algunas imágenes de ejemplo
plt.figure(figsize=(8, 3))
for i in range(12):
    plt.subplot(2, 6, i+1)
    plt.imshow(x_train[i], cmap="gray")
    plt.title(f"Etiqueta: {y_train[i]}")
    plt.axis("off")
plt.suptitle("Ejemplos de MNIST")
plt.tight_layout()
plt.show()

Keras está descargando el dataset MNIST desde los servidores de Google.

Tamaño: 11,490,434 bytes (~11 MB).

Como ya se descargó completo, la próxima vez se cargará desde la caché y no lo descargará de nuevo.

Formato de los datos
Train: (60000, 28, 28) (60000,)
Test : (10000, 28, 28) (10000,)


Esto describe las formas (shapes) de los arrays que contiene MNIST:

Train: (60000, 28, 28)

60,000 imágenes de entrenamiento.

Cada imagen tiene tamaño 28×28 pixeles en blanco y negro.

(60000,)

Son las etiquetas (labels) de esas 60,000 imágenes.

Cada número es un dígito entre 0 y 9.

Test: (10000, 28, 28)

10,000 imágenes de prueba.

También de tamaño 28×28.

(10000,)

Etiquetas correspondientes a esas 10,000 imágenes.

 En palabras simples:

Tenemos 60,000 imágenes para entrenar la red y 10,000 imágenes para probarla.

Cada imagen es un número escrito a mano (0–9) en formato de matriz 28×28 pixeles.

Las etiquetas (y) nos dicen cuál es el dígito real en cada imagen.

Por eso, en el gráfico se ven ejemplos con la etiqueta arriba (Etiqueta: 5, Etiqueta: 0, etc.), que corresponde a la clase correcta de cada imagen.

In [ ]:
# 3) Preprocesamiento (normalización y reshape)
# @title Preprocesamiento: normalización [0,1] y reshape a (28,28,1)
# Escalamos a [0,1] para que el entrenamiento sea más estable
x_train = x_train.astype("float32") / 255.0
x_test  = x_test.astype("float32") / 255.0

# Añadir canal (grayscale) -> (alto, ancho, canales)
x_train = np.expand_dims(x_train, -1)  # (60000, 28, 28, 1)
x_test  = np.expand_dims(x_test, -1)   # (10000, 28, 28, 1)

# Dividir un conjunto de validación desde train (por ejemplo, 10k imágenes)
x_val, y_val = x_train[-10000:], y_train[-10000:]
x_train, y_train = x_train[:-10000], y_train[:-10000]

print("x_train:", x_train.shape, "y_train:", y_train.shape)
print("x_val  :", x_val.shape,   "y_val  :", y_val.shape)
print("x_test :", x_test.shape,  "y_test :", y_test.shape)

Esa salida corresponde al estado del dataset después de hacer el preprocesamiento y la división en entrenamiento, validación y prueba. Vamos por partes:

🔹 1) x_train: (50000, 28, 28, 1)

Son 50,000 imágenes para entrenar el modelo.

Cada imagen es de tamaño 28×28 pixeles.

El último 1 significa que tienen 1 canal → imágenes en blanco y negro (si fueran a color tendrían 3 canales: RGB).

En palabras simples: 50 mil imágenes de entrenamiento en escala de grises.

🔹 2) y_train: (50000,)

Son las etiquetas reales (números 0–9) para esas 50,000 imágenes de entrenamiento.

El shape (50000,) significa que hay 50,000 valores enteros, uno por imagen.

🔹 3) x_val: (10000, 28, 28, 1)

Son 10,000 imágenes de validación.

El modelo las usa para ajustar hiperparámetros y evitar sobreajuste.

Igual que x_train, cada imagen es de 28×28 con 1 canal.

🔹 4) y_val: (10000,)

Etiquetas correspondientes a esas 10,000 imágenes de validación.

🔹 5) x_test: (10000, 28, 28, 1)

Son 10,000 imágenes de prueba.

Se usan al final para medir el rendimiento real del modelo en datos nunca vistos.

🔹 6) y_test: (10000,)

Etiquetas de esas 10,000 imágenes de prueba.

En resumen para tus estudiantes:

Entrenamiento → 50,000 imágenes.

Validación → 10,000 imágenes.

Prueba → 10,000 imágenes.

Cada imagen = matriz de 28×28 pixeles en blanco y negro.

y son las etiquetas que dicen cuál número representa la imagen.

In [ ]:
# 4) (Opcional) Aumentación de datos
# @title Data augmentation (opcional pero útil)
# Pequeñas transformaciones que ayudan a generalizar (evitar sobreajuste)
data_augment = keras.Sequential([
    layers.RandomRotation(0.08),
    layers.RandomTranslation(0.05, 0.05),
    layers.RandomZoom(0.08),
])

¿Qué es la aumentación de datos?

Es una técnica donde, a partir de las imágenes originales de entrenamiento, se generan versiones ligeramente modificadas.

El objetivo es que la red vea más variedad y aprenda a generalizar mejor, evitando el sobreajuste (memorizar en lugar de aprender patrones).

 Lo que hace cada transformación en tu código:
data_augment = keras.Sequential([
    layers.RandomRotation(0.08),
    layers.RandomTranslation(0.05, 0.05),
    layers.RandomZoom(0.08),
])


layers.RandomRotation(0.08)

Gira la imagen un ángulo aleatorio (máximo 0.08 radianes ≈ 4.5 grados).

Ejemplo: un número "5" puede verse un poco inclinado.

layers.RandomTranslation(0.05, 0.05)

Mueve la imagen un poco en x e y (máximo 5% de su tamaño).

Ejemplo: un "3" puede estar ligeramente desplazado hacia la derecha o hacia arriba.

layers.RandomZoom(0.08)

Aplica un zoom aleatorio (acercar o alejar hasta un 8%).

Ejemplo: un "8" puede verse un poco más grande o más pequeño.

¿Por qué es útil?

Los datos originales de MNIST ya son buenos, pero en problemas reales, las imágenes no siempre estarán centradas o perfectamente alineadas.

Con la aumentación, la red neuronal aprende a reconocer el dígito sin importar si está un poquito girado, movido o ampliado.

Esto ayuda a que el modelo tenga mejor rendimiento en datos nuevos.

 Ejemplo:
Imagina que solo tienes una foto de un gato. Si la giras un poco, haces zoom o la mueves, sigue siendo un gato.
Si entrenas a la red con esas variaciones, aprenderá a reconocer gatos en muchas posiciones, no solo en la original.

In [ ]:
# 5) Definir la CNN
# @title Modelo CNN sencillo para MNIST
# Arquitectura: Conv -> ReLU -> Pool -> Conv -> ReLU -> Pool -> Dense -> Dropout -> Softmax
inputs = keras.Input(shape=(28, 28, 1))

x = data_augment(inputs)          # Aplicar aumentación en tiempo de entrenamiento
x = layers.Conv2D(32, 3, padding="same", activation="relu")(x)
x = layers.MaxPooling2D()(x)
x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
x = layers.MaxPooling2D()(x)

x = layers.Flatten()(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.3)(x)        # Dropout para reducir sobreajuste
outputs = layers.Dense(10, activation="softmax")(x)  # 10 clases (0-9)

model = keras.Model(inputs, outputs, name="cnn_mnist")
model.summary()

Nombre del modelo que definiste (cnn_mnist).

 Capas del modelo
1. Entrada
input_layer (InputLayer) → (None, 28, 28, 1)


Entrada: imágenes de 28×28 píxeles con 1 canal (escala de grises).

None significa que puede recibir cualquier tamaño de batch (ej. 32, 128 imágenes por lote).

2. Data augmentation (opcional)
sequential (Sequential) → (None, 28, 28, 1)


Es el bloque donde aplicas rotación, traslación y zoom aleatorios.

La salida sigue teniendo el mismo tamaño (28×28×1).

3. Primera convolución
conv2d (Conv2D) → (None, 28, 28, 32)
Param # = 320


Filtros: 32 filtros de 3×3.

Cada filtro aprende a detectar patrones locales (bordes, curvas, trazos).

Cálculo parámetros:
(3×3×1+1)×32=320

(9 pesos por filtro × 1 canal + 1 bias) × 32 filtros.

 Salida: 32 mapas de características de tamaño 28×28.

4. MaxPooling
max_pooling2d (MaxPooling2D) → (None, 14, 14, 32)


Reduce la resolución a la mitad (de 28×28 a 14×14).

Mantiene los 32 filtros.

Sin parámetros entrenables.

 Ayuda a reducir cómputo y sobreajuste.

5. Segunda convolución
conv2d_1 (Conv2D) → (None, 14, 14, 64)
Param # = 18,496


Filtros: 64 filtros de 3×3.

Cálculo:
(3×3×32+1)×64=18,496

 Ahora detecta patrones más complejos (formas compuestas a partir de bordes).

6. MaxPooling 2
max_pooling2d_1 → (None, 7, 7, 64)


Reduce de 14×14 a 7×7.

Mantiene 64 mapas de características.

7. Flatten
flatten → (None, 3136)


Convierte el tensor 7×7×64 en un vector plano de 3136 valores por imagen.

Esto conecta la parte convolucional con las capas densas.

8. Capa densa (oculta)
dense → (None, 128)
Param # = 401,536

128 neuronas totalmente conectadas.

Cálculo parámetros:
3136×128+128=401,536

 Aprende combinaciones globales de las características extraídas por las convoluciones.

9. Dropout
dropout → (None, 128)


Apaga aleatoriamente un porcentaje de neuronas en cada paso de entrenamiento.

Sirve para evitar sobreajuste.

No tiene parámetros.

10. Capa de salida
dense_1 → (None, 10)
Param # = 1,290


10 neuronas → una por cada dígito (0 al 9).

Activación: softmax, que devuelve probabilidades que suman 1.

Cálculo parámetros:

128×10+10=1,290
 Totales
Total params: 421,642 (1.61 MB)
Trainable params: 421,642
Non-trainable params: 0


421,642 parámetros entrenables → los pesos que la red ajusta durante el entrenamiento.

Ocupa apenas ~1.6 MB, lo cual es muy pequeño comparado con redes modernas (que llegan a millones de parámetros).

Resumen para explicar en clase:

La CNN primero extrae características locales con convoluciones (bordes, formas).

Luego reduce la dimensionalidad con pooling.

Convierte todo en un vector con Flatten.

Las capas densas hacen la clasificación final.

La salida softmax da una probabilidad por cada dígito (0–9).

In [ ]:
# 6) Compilar y entrenar con EarlyStopping
# @title Compilación y entrenamiento
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",  # etiquetas enteras 0..9
    metrics=["accuracy"]
)

early = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True
)

history = model.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=30,
    batch_size=128,
    callbacks=[early],
    verbose=1
)

Esa parte es el registro del entrenamiento de tu modelo. Cada vez que lo ves (Epoch 1, 2, 3…), corresponde a una época.

 ¿Qué es una época?

Una época es cuando la red neuronal ve todo el conjunto de entrenamiento una vez completo.

En tu caso, se definieron 30 épocas → por eso lo intenta hasta 30 veces (aunque puede detenerse antes si usas EarlyStopping).

 Desglose de la salida que ves (ejemplo con Epoch 1)
Epoch 1/30
391/391 ━━━━━━━━━━━━━━━━━━━━ 68s 158ms/step - accuracy: 0.7299 - loss: 0.8188 - val_accuracy: 0.9748 - val_loss: 0.0837

1. Epoch 1/30

Estamos en la época 1 de 30 posibles.

2. 391/391

El entrenamiento se divide en batches (lotes).

Como tu batch_size fue 128 y tenías ~50,000 imágenes de entrenamiento → eso da 391 lotes.

Significa que ya terminó todos los lotes de esa época.

3. 68s 158ms/step

Tiempo que tardó en completar esa época: 68 segundos, con 158 ms promedio por lote.

4. accuracy: 0.7299

Precisión del modelo en el conjunto de entrenamiento durante esa época.

Ejemplo: en la época 1, acertó el 72.99% de las veces.

5. loss: 0.8188

Pérdida (error) en el conjunto de entrenamiento.

Mientras menor sea el loss, mejor está ajustando la red.

6. val_accuracy: 0.9748

Precisión en el conjunto de validación → aquí fue 97.48%.

Es más alta que la de entrenamiento inicial porque las CNN con MNIST aprenden rápido.

7. val_loss: 0.0837

Error en validación.

Si este valor baja, significa que la red está aprendiendo a generalizar.

 Evolución en las siguientes épocas

Epoch 2: accuracy: 0.9473, val_accuracy: 0.9838 → mejora clara, ya casi 95% en entrenamiento y 98% en validación.

Epoch 3–5: accuracy sigue subiendo (95–97%) y val_accuracy se mantiene alto (~98–99%).

loss y val_loss bajan → el modelo está aprendiendo cada vez mejor.

 En resumen para explicar en clase:

Cada época = el modelo ve todo el dataset una vez.

Dentro de cada época hay muchos steps/lotes (391 en este caso).

Se reporta el desempeño en entrenamiento (accuracy, loss) y en validación (val_accuracy, val_loss).

Lo que queremos: accuracy alto y loss bajo, tanto en train como en val.

Si train_accuracy sube pero val_accuracy baja → hay sobreajuste.

In [ ]:
# 7) Curvas de entrenamiento (loss y accuracy)
# @title Curvas de entrenamiento
plt.figure()
plt.plot(history.history["loss"], label="train_loss")
plt.plot(history.history["val_loss"], label="val_loss")
plt.xlabel("Época"); plt.ylabel("Pérdida"); plt.title("Pérdida: train vs val")
plt.legend(); plt.grid(True); plt.show()

plt.figure()
plt.plot(history.history["accuracy"], label="train_acc")
plt.plot(history.history["val_accuracy"], label="val_acc")
plt.xlabel("Época"); plt.ylabel("Accuracy"); plt.title("Accuracy: train vs val")
plt.legend(); plt.grid(True); plt.show()

1) Pérdida (loss): train vs val

Eje X: número de épocas (iteraciones de entrenamiento completas).

Eje Y: valor de la pérdida (loss), que mide el error del modelo.

 Interpretación del gráfico:

La línea azul (train_loss) empieza alta y baja rápidamente → el modelo aprende a ajustar los pesos para reducir error en el conjunto de entrenamiento.

La línea naranja (val_loss) también baja y se mantiene baja → significa que el modelo generaliza bien en datos nuevos (validación).

Ambas líneas descienden de forma estable y nunca se separan demasiado → no hay sobreajuste importante.

 El modelo está aprendiendo bien, los errores bajan tanto en entrenamiento como en validación.

📈 2) Precisión (accuracy): train vs val

Eje X: épocas.

Eje Y: precisión (proporción de aciertos).

 Interpretación del gráfico:

La línea azul (train_acc) sube rápido desde ~86% hasta casi 99%.

La línea naranja (val_acc) empieza ya muy alta (~97%) y llega casi al 99.5%.

El hecho de que val_acc esté ligeramente por encima de train_acc es normal en MNIST → el dataset es sencillo y la red con regularización (dropout + data augmentation) generaliza mejor que lo que memoriza en train.

 Resultado: el modelo logra una precisión altísima (>99%) en validación y se mantiene estable → entrenó de forma excelente.

 Resumen:

Loss vs Epoch: el error del modelo bajó tanto en entrenamiento como en validación → aprendió de forma consistente.

Accuracy vs Epoch: la precisión alcanzó casi 99% en validación → el modelo reconoce dígitos manuscritos con gran exactitud.

Como las curvas de train y val se comportan parecidas, no hubo sobreajuste.

 En otras palabras: tu modelo CNN está aprendiendo muy bien y generalizando casi perfecto en MNIST

In [ ]:
# 8) Evaluación en test + matriz de confusión y reporte
# @title Evaluación en test + métricas detalladas
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"Accuracy en test: {test_acc:.4f} | Pérdida en test: {test_loss:.4f}")

# Predicciones de probabilidad (10 clases) para todo test
y_prob = model.predict(x_test, verbose=0)
y_pred = np.argmax(y_prob, axis=1)

# Matriz de confusión
cm = confusion_matrix(y_test, y_pred)
print("Matriz de confusión:\n", cm)

# Reporte de clasificación por clase (precision/recall/F1)
print("\nReporte de clasificación:\n",
      classification_report(y_test, y_pred, digits=4))

1) Accuracy y pérdida en test
Accuracy en test: 0.9934 | Pérdida en test: 0.0232


Accuracy (0.9934 = 99.34%) → de cada 100 imágenes, el modelo clasifica bien aproximadamente 99.3.

Pérdida (loss = 0.0232) → mide el error; es muy bajo, lo que confirma que el modelo está muy bien entrenado.

 Resultado global: el modelo tiene un desempeño excelente en datos nunca vistos.

 2) Matriz de confusión

Cada fila = clase real, cada columna = clase predicha.

Ejemplo:

Fila 0 (dígitos "0"): había 980 imágenes.

977 fueron clasificadas correctamente como "0".

1 como "2", 1 como "6", 1 como "9".

Algunas observaciones:

La mayoría de las entradas son diagonales con valores grandes → casi todas las imágenes fueron clasificadas correctamente.

Los errores son muy pocos y aislados, por ejemplo:

Un par de "5" confundidos como "3".

Algunos "9" confundidos con "4" o "5".

Esto tiene sentido porque algunos dígitos manuscritos se parecen.

 Interpretación: la red solo se equivoca en casos donde la escritura es confusa.

 3) Reporte de clasificación

Incluye 3 métricas por clase:

Precision (precisión): de los dígitos que el modelo dijo “esto es un 5”, ¿cuántos eran realmente 5?

Recall (exhaustividad): de todos los 5 que había, ¿cuántos encontró el modelo?

F1-score: balance entre precision y recall.

Support: cantidad de ejemplos reales de cada clase en el test.

Ejemplo clase 0:

Precision = 0.999 → casi todas las veces que dijo "0", acertó.

Recall = 0.9969 → de 980 ceros, encontró el 99.69%.

F1 = 0.998 → excelente balance.

 Todas las clases tienen métricas entre 98.6% y 100%, lo cual es altísimo.

 4) Totales
accuracy = 0.9934
macro avg = 0.9934
weighted avg = 0.9934


Accuracy global: 99.34%.

Macro avg: promedio simple de todas las clases (trata todas por igual).

Weighted avg: promedio ponderado por cantidad de muestras por clase.

 Como MNIST tiene las clases balanceadas, los tres coinciden.

 Resumen para tus estudiantes:

El modelo logra 99.34% de aciertos en el test, lo cual es casi perfecto.

Los errores ocurren en casos difíciles, como confundir un “9” con un “4” o un “5”.

Las métricas de cada clase son excelentes (todas ≥ 98.6%).

La red convolucional aprendió a reconocer dígitos manuscritos de forma muy confiable.

In [ ]:
# 9) Visualizar predicciones individuales (imagen + top probabilidades)
# @title Predicción visual de algunos ejemplos
def mostrar_prediccion(idx):
    img = x_test[idx].squeeze()      # (28,28)
    true_label = y_test[idx]
    probs = y_prob[idx]              # vector de 10 probabilidades
    pred_label = np.argmax(probs)

    plt.figure(figsize=(10,3))
    # Imagen
    plt.subplot(1, 2, 1)
    plt.imshow(img, cmap="gray")
    plt.title(f"Etiqueta real: {true_label} | Predicha: {pred_label}")
    plt.axis("off")

    # Barras con probabilidades
    plt.subplot(1, 2, 2)
    plt.bar(np.arange(10), probs)
    plt.xticks(np.arange(10))
    plt.ylim(0, 1.0)
    plt.title("Probabilidades por clase")
    plt.grid(True, axis="y", linestyle="--", alpha=0.5)
    plt.show()

# Probar con algunas imágenes de test
for i in [0, 1, 2, 3, 4]:
    mostrar_prediccion(i)
# Por qué CNN para imágenes: las convoluciones aprenden filtros (bordes, esquinas, trazos) y los combinan para reconocer dígitos.
# Preprocesamiento: normalizar a [0,1] mejora la estabilidad del entrenamiento.
# Aumentación de datos: pequeñas rotaciones/traslaciones/zooms ayudan a generalizar.
# Arquitectura típica: Conv → ReLU → Pool (repetir), luego Dense y Softmax.
# Métricas: accuracy global; revisar la matriz de confusión para ver qué dígitos confunde.
# Predicciones: la salida softmax son probabilidades por clase; elegimos la de mayor probabilidad.

) Imagen a la izquierda

Es una de las imágenes del conjunto de prueba (un dígito manuscrito).

Encima aparece la etiqueta real (ground truth) y la clase predicha por la red.
Ejemplo:

Etiqueta real: 7 | Predicha: 7


 Significa que el dígito realmente era un 7 y el modelo también lo clasificó como 7 (acertó).

 2) Gráfico de barras a la derecha

Representa las probabilidades asignadas a cada clase (0–9) por la última capa softmax.

La barra más alta indica la clase que el modelo considera más probable.

Como en tus ejemplos la barra está pegada a 1.0, significa que el modelo estaba muy seguro de su predicción.

 Ejemplo 1
Etiqueta real: 7 | Predicha: 7


Imagen de un "7".

Softmax asigna probabilidad ≈ 1.0 al número 7 y 0.0 al resto.
 Clasificación correcta y con máxima confianza.

 Ejemplo 2
Etiqueta real: 2 | Predicha: 2


Imagen de un "2".

Probabilidad ≈ 1.0 en la clase 2.
 Correcto y seguro.

 Ejemplo 3
Etiqueta real: 0 | Predicha: 0


Imagen de un "0".

Probabilidad ≈ 1.0 en la clase 0.
 Correcto y seguro.

 Interpretación general

La red no solo acierta la clase correcta, sino que lo hace con altísima confianza (probabilidades cercanas a 100%).

Esto refleja lo que ya viste en el accuracy global del 99% → el modelo CNN entrenado en MNIST es muy fiable para reconocer dígitos manuscritos.

Si en algún caso hubiera confusión, verías barras repartidas entre dos números (ej. un “9” que también parece un “4”).

 En pocas palabras:
Estos gráficos muestran cómo la red “piensa”: primero procesa la imagen, luego asigna probabilidades a todas las clases, y finalmente elige la más alta como predicción.

In [ ]:
# Ejemplo 4
# @title Preparación e importaciones
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.metrics import confusion_matrix, classification_report

import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)

In [ ]:
# dataset IMDB (inglés, 25k train / 25k test)
# @title Cargar IMDB ya tokenizado por Keras (palabras -> índices)
# num_words: tamaño del vocabulario (se quedará con las 10k palabras más frecuentes)
vocab_size = 10000
(x_train, y_train), (x_test, y_test) = keras.datasets.imdb.load_data(num_words=vocab_size)

print("train:", x_train.shape, y_train.shape)
print("test :", x_test.shape, y_test.shape)

# Longitudes de reseñas (para dimensionar el padding)
lens = [len(seq) for seq in x_train]
print("Longitud media:", int(np.mean(lens)), "| p90:", int(np.percentile(lens, 90)))

) Descarga del dataset
Downloading data from https://storage.googleapis.com/tensorflow/tf-keras-datasets/imdb.npz
17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Está descargando el archivo imdb.npz (≈ 17.4 MB) desde los servidores de Google.

Una vez descargado, se guarda en caché, así que la próxima vez no vuelve a bajarlo.
) ¿Qué es el archivo imdb.npz?

Es un archivo comprimido en formato NumPy .npz.

Contiene el dataset IMDB de reseñas de películas (Internet Movie Database).

Son 50,000 reseñas de películas en inglés, ya preprocesadas en forma de secuencias de enteros.

 Cada entero corresponde a una palabra en el vocabulario (ej. 4 = "the", 20 = "film").

 2) Variables principales que contiene

Cuando lo cargas con:

(x_train, y_train), (x_test, y_test) = keras.datasets.imdb.load_data()


se generan estas variables:

x_train

Tipo: lista de listas (secuencias).

Cada elemento es una reseña representada como una secuencia de enteros (tokens de palabras).

Ejemplo:

x_train[0] = [1, 14, 22, 16, 43, 530, 973, ...]


El vocabulario está ordenado por frecuencia de palabras.

Por defecto, se limitan a las num_words más frecuentes (ej. 10,000).

y_train

Tipo: array de enteros.

Etiquetas binarias:

0 = reseña negativa

1 = reseña positiva

Ejemplo:

y_train[0] = 1   # esta reseña es positiva


x_test

Igual que x_train, pero contiene las 25,000 reseñas de prueba.

y_test

Igual que y_train, pero con las etiquetas de las reseñas de prueba.

 3) Tipos de variables

x_train, x_test:

Tipo: listas de NumPy (objetos tipo list o ndarray de Python).

Contienen secuencias de enteros de longitud variable (cada reseña puede tener distinto largo).

y_train, y_test:

Tipo: ndarray de enteros (int64).

Contienen etiquetas binarias (0 o 1).

🔹 2) Tamaño de los conjuntos
train: (25000,) (25000,)
test : (25000,) (25000,)


train: (25000,) (25000,) → el dataset de entrenamiento tiene 25,000 reseñas y 25,000 etiquetas.

El primer (25000,) son las reseñas (cada reseña es una lista de enteros que representan palabras).

El segundo (25000,) son las etiquetas (0 = reseña negativa, 1 = positiva).

test: (25000,) (25000,) → lo mismo pero para las reseñas de prueba.

 En total: 50,000 reseñas de películas, la mitad positivas y la mitad negativas.

🔹 3) Estadísticas de longitud
Longitud media: 238 | p90: 467


Longitud media (238) → en promedio, cada reseña tiene 238 palabras (tokens).

p90 (percentil 90 = 467) → el 90% de las reseñas tiene 467 palabras o menos.

Esto sirve para decidir el maxlen en el padding (ej. 200, 300, 500).

Ejemplo: si elegimos maxlen=200, estaremos truncando algunas reseñas largas, pero capturamos la mayoría del texto.

En resumen para explicar a tus estudiantes:

Se descargó el dataset IMDB (50k reseñas).

Cada reseña ya está convertida en una lista de números (índices de palabras).

Hay 25k reseñas para entrenar y 25k para probar.

En promedio, las reseñas son de 238 palabras, aunque algunas llegan a 467 o má

In [ ]:
# 3) Padding (alinear secuencias a una misma longitud)
# @title Padding de secuencias
# Definimos una longitud fija (p. ej., 200). Reseñas más largas se recortan, más cortas se rellenan con 0
max_len = 200

x_train_pad = keras.preprocessing.sequence.pad_sequences(x_train, maxlen=max_len, padding="post", truncating="post")
x_test_pad  = keras.preprocessing.sequence.pad_sequences(x_test,  maxlen=max_len, padding="post", truncating="post")

# Creamos un conjunto de validación (p. ej. 20% de train)
val_split = 0.2
n_val = int(len(x_train_pad) * val_split)

x_val_pad, y_val = x_train_pad[:n_val], y_train[:n_val]
x_tr_pad,  y_tr  = x_train_pad[n_val:], y_train[n_val:]

print("Train:", x_tr_pad.shape,  y_tr.shape)
print("Val  :", x_val_pad.shape, y_val.shape)
print("Test :", x_test_pad.shape, y_test.shape)

Las reseñas de IMDB vienen como listas de enteros (tokens de palabras).

Cada reseña tiene longitud distinta: unas son cortas (10–20 palabras), otras muy largas (500+ palabras).

Las redes neuronales requieren que todas las entradas tengan el mismo tamaño para poder procesarlas en lotes (batch).

Solución:

Definimos una longitud fija (en este caso, 200).

Si una reseña es más larga que 200 tokens, se recorta.

Si es más corta que 200 tokens, se rellena con ceros (0) al final → esto es el padding.

2) Lo que significa la salida
Train: (20000, 200) (20000,)
Val  : (5000, 200) (5000,)
Test : (25000, 200) (25000,)


Train: (20000, 200)

20,000 reseñas de entrenamiento.

Cada reseña ahora es un vector de 200 enteros (tokens + ceros de relleno si hacía falta).

(20000,) → 20,000 etiquetas (0 = negativo, 1 = positivo).

Val : (5000, 200)

5,000 reseñas de validación.

Cada reseña también tiene 200 tokens tras el padding.

(5000,) → 5,000 etiquetas (clases binarias).

Test : (25000, 200)

25,000 reseñas de prueba.

Todas normalizadas a longitud 200.

(25000,) → 25,000 etiquetas de prueba.

3) Ejemplo sencillo

Supongamos que maxlen = 6 y tenemos estas 3 reseñas tokenizadas:

Reseña 1: [12, 35, 7]          # longitud 3
Reseña 2: [4, 9, 16, 2, 87, 1] # longitud 6
Reseña 3: [5, 22, 13, 8, 19]   # longitud 5


Después del padding → todas tienen longitud 6:

Reseña 1: [12, 35, 7, 0, 0, 0]
Reseña 2: [4, 9, 16, 2, 87, 1]
Reseña 3: [5, 22, 13, 8, 19, 0]


En resumen para tus estudiantes:

El padding convierte todas las reseñas en secuencias del mismo largo (200 en este caso).

Esto permite entrenar la red neuronal en lotes sin problemas.

Los 0 agregados no representan palabras reales, solo son relleno.

In [ ]:
# 4) Modelo LSTM (Embedding → LSTM → Dense sigmoide)
# @title Definir LSTM para sentimiento binario
embedding_dim = 64   # tamaño del embedding (vector que representa cada palabra)
lstm_units    = 64

model = keras.Sequential([
    layers.Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len),
    layers.SpatialDropout1D(0.2),           # ayuda a robustez ante sobreajuste en embeddings
    layers.LSTM(lstm_units, return_sequences=False),  # LSTM "lee" la reseña
    layers.Dropout(0.3),
    layers.Dense(1, activation="sigmoid")    # salida binaria: 0 (neg), 1 (pos)
])

model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

1) La advertencia (UserWarning)
Argument `input_length` is deprecated. Just remove it.


Antes en Keras se usaba input_length=max_len en la capa Embedding.

Ahora ya no hace falta especificarlo: Keras lo detecta automáticamente cuando se le pasan datos.
Solo es un aviso, no es error. Puedes dejarlo o quitarlo sin problema.

2) Resumen del modelo (model.summary())

Por ahora te muestra:

Model: "sequential"
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
...
└─────────────────────────────────┴────────────────────────┴───────────────┘
 Total params: 0 (0.00 B)
 Trainable params: 0 (0.00 B)
 Non-trainable params: 0 (0.00 B)


¿Por qué aparecen ? y 0 parámetros?
Porque el modelo todavía no ha sido “build” (construido).

Keras necesita ver la forma real de los datos de entrada para calcular cuántos parámetros va a tener cada capa.

Una vez que haces el primer model.fit() o model.build((None, max_len)), se actualizará con los tamaños correctos.

3) Explicación de cada capa en tu modelo

Embedding (Embedding layer)

Convierte cada palabra (entero/token) en un vector denso de tamaño embedding_dim = 64.

Ejemplo: la palabra con índice 27 → [0.12, -0.34, 0.55, ...] (64 números).

Parámetros = vocab_size × embedding_dim.

SpatialDropout1D

Apaga aleatoriamente vectores de palabras completos durante el entrenamiento.

Previene sobreajuste, especialmente útil en NLP.

LSTM (Long Short-Term Memory)

Lee la secuencia de embeddings palabra por palabra.

Retiene contexto pasado y aprende dependencias a lo largo del texto.

Número de parámetros depende de embedding_dim y lstm_units.

Dropout

Apaga algunas conexiones de la LSTM en cada paso.

Ayuda a regularizar.

Dense (sigmoid)

Capa final de 1 neurona con activación sigmoide.

Salida = probabilidad entre 0 y 1 → 0 (negativo), 1 (positivo).

4) ¿Por qué Total params = 0?

Porque aún no se construyó el grafo con la forma de los datos.
En cuanto entrenes o hagas un model.build((None, max_len)), aparecerá algo así:

Layer (type)     Output Shape      Param #
Embedding        (None, 200, 64)   640000   # si vocab_size=10000
LSTM             (None, 64)        33024
Dense            (None, 1)         65
Total params: ~673,089


En resumen:

El resumen con parámetros 0 solo significa que el modelo aún no se “ha usado”.

Una vez que empieces el entrenamiento, Keras calculará automáticamente cuántos parámetros tiene cada capa.

Cada capa del modelo cumple un rol: Embedding → Dropout → LSTM → Dropout → Dense(sigmoide) para clasificación binaria de sentimiento.

In [ ]:
# 5) Entrenamiento con EarlyStopping
# @title Entrenamiento
early = keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)

history = model.fit(
    x_tr_pad, y_tr,
    validation_data=(x_val_pad, y_val),
    epochs=12,
    batch_size=128,
    callbacks=[early],
    verbose=1
)

1) EarlyStopping
early = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=3, restore_best_weights=True
)


monitor="val_loss" → observa la pérdida (loss) en el conjunto de validación.

patience=3 → si durante 3 épocas seguidas no mejora el val_loss, detiene el entrenamiento.

restore_best_weights=True → al final restaura los pesos de la mejor época (no de la última).

Esto evita que el modelo se sobreentrene y empiece a memorizar en lugar de generalizar.

2) Cada época del log
Epoch 1/12
accuracy: 0.5213 - loss: 0.6896 - val_accuracy: 0.5538 - val_loss: 0.6736


En el entrenamiento, el modelo acierta 52.13% (prácticamente azar, ya que el dataset es binario).

En validación, acierta un poco mejor (55.38%) y la pérdida es 0.6736.
Normal en la primera época: la red apenas empieza a ajustar.

Epoch 2/12
accuracy: 0.5554 - loss: 0.6753 - val_accuracy: 0.5030 - val_loss: 0.6916


Entrenamiento mejora un poco (55.54%) y pérdida baja.

Pero en validación cae fuerte (50.3%, casi azar) y la pérdida sube.
El modelo no está generalizando bien aún.

Epoch 3/12
accuracy: 0.5337 - loss: 0.6921 - val_accuracy: 0.4932 - val_loss: 0.6969


Entrenamiento baja otra vez (53.37%).

Validación empeora (49.32%), y val_loss sube más (0.6969).
Señal de que el modelo está teniendo dificultades para aprender (probablemente necesita más ajustes de hiperparámetros o vocabulario).

Epoch 4/12
accuracy: 0.5144 - loss: 0.6949 - val_accuracy: 0.5610 - val_loss: 0.6798


Entrenamiento se queda casi en azar (51.44%).

Validación sube otra vez a 56.10%, val_loss mejora un poco (0.6798).
Parece que el modelo está inestable: a veces mejora validación, a veces empeora.

3) Interpretación global

El modelo está rondando entre 50–56% de accuracy, lo cual es bajo (casi como lanzar una moneda).

La pérdida en validación (val_loss) no mejora consistentemente, por eso EarlyStopping terminará deteniendo el entrenamiento.

Posibles causas:

Longitud de secuencia (max_len) demasiado corta o larga → puede estar truncando frases importantes o agregando mucho relleno (0s).

Embedding_dim pequeño (64) → quizá necesite mayor dimensionalidad (100–128).

LSTM units pocas (64) → podría necesitar más unidades o capas.

Dataset IMDB crudo es complejo: entrenar bien requiere más épocas, regularización y/o embeddings preentrenados.

En resumen:

Cada época muestra cómo el modelo aprende (accuracy/loss en train) y cómo generaliza (val_accuracy/val_loss).

Aquí vemos que el modelo aprende lento y de manera inestable.

EarlyStopping evita perder tiempo si el modelo deja de mejorar.

In [ ]:
# 6) Curvas de pérdida y accuracy
# @title Curvas de entrenamiento
plt.figure()
plt.plot(history.history["loss"], label="train_loss")
plt.plot(history.history["val_loss"], label="val_loss")
plt.xlabel("Época"); plt.ylabel("Pérdida"); plt.title("Pérdida: train vs val")
plt.legend(); plt.grid(True); plt.show()

plt.figure()
plt.plot(history.history["accuracy"], label="train_acc")
plt.plot(history.history["val_accuracy"], label="val_acc")
plt.xlabel("Época"); plt.ylabel("Accuracy"); plt.title("Accuracy: train vs val")
plt.legend(); plt.grid(True); plt.show()

Gráfico 1: Pérdida (loss) – train vs val

Eje X (horizontal): número de épocas (iteraciones completas sobre los datos).

Eje Y (vertical): valor de la pérdida (qué tan mal predice el modelo; más bajo = mejor).

Línea azul (train_loss): error en el conjunto de entrenamiento.

Línea naranja (val_loss): error en el conjunto de validación.

Interpretación:

En la época 1, ambas pérdidas bajan un poco (aprendizaje inicial).

Después (época 2 en adelante), la pérdida de validación sube → el modelo no está generalizando bien.

La de entrenamiento sube también, lo cual indica que la red no está aprendiendo de forma estable (no converge).

Gráfico 2: Accuracy – train vs val

Eje X: épocas.

Eje Y: accuracy (porcentaje de aciertos; más alto = mejor).

Línea azul (train_acc): precisión en entrenamiento.

Línea naranja (val_acc): precisión en validación.

Interpretación:

En la época 1, el modelo alcanza ~58% en train y ~55% en validación (mejor que azar, que sería 50%).

Pero en las épocas siguientes:

train_acc baja ligeramente.

val_acc cae por debajo de 50% y recién sube un poco en la última época.

Esto indica que el modelo no encuentra un patrón sólido en los datos.

Conclusión de ambos gráficos

El modelo no está convergiendo bien: las curvas son inestables.

Accuracy está alrededor de 50–55%, lo cual es casi azar para un problema binario.

Pérdida de validación sube en lugar de bajar → señal de problema de aprendizaje (modelo muy pequeño, embeddings poco expresivos, o hiperparámetros no adecuados).

Recomendaciones para mejorar

Aumentar el embedding_dim (ej. de 64 → 128 o 200).

Usar más unidades LSTM (ej. 128 o incluso 2 capas apiladas).

Probar con Bidirectional LSTM, que captura contexto hacia adelante y atrás.

Revisar el max_len del padding (si es demasiado corto, se pierde información; demasiado largo mete mucho ruido).

Usar embeddings preentrenados (como GloVe).

In [ ]:
# 7) Evaluación en test + métricas detalladas
# @title Evaluación en test + métricas
test_loss, test_acc = model.evaluate(x_test_pad, y_test, verbose=0)
print(f"Accuracy en test: {test_acc:.4f} | Pérdida en test: {test_loss:.4f}")

# Probabilidades y clases predichas
y_prob = model.predict(x_test_pad, verbose=0).reshape(-1)
y_pred = (y_prob >= 0.5).astype(int)

# Matriz de confusión y reporte
print("Matriz de confusión:\n", confusion_matrix(y_test, y_pred))
print("\nReporte de clasificación:\n", classification_report(y_test, y_pred, digits=4))

1) Accuracy y pérdida en test
Accuracy en test: 0.5595 | Pérdida en test: 0.6731


Accuracy = 55.95% → El modelo acierta poco más de la mitad de las veces.

Loss = 0.6731 → La pérdida es bastante alta (recuerda que para clasificación binaria idealmente debería estar <0.5).

Conclusión: el modelo no está aprendiendo bien, se comporta casi como azar (50%).

2) Matriz de confusión
 [[12003   497]
 [10516  1984]]


Filas = clase real.

Columnas = clase predicha.

Interpretación:

Clase 0 (negativa):

Correctas: 12,003

Incorrectas: 497

Clase 1 (positiva):

Correctas: 1,984

Incorrectas: 10,516

El modelo tiende a clasificar casi todo como negativo (0), por eso acierta mucho en negativos pero falla en la mayoría de positivos.

3) Reporte de clasificación
precision    recall  f1-score   support

Clase 0 (negativa)

Precision = 0.5330 → De todas las reseñas que predijo como negativas, el 53% realmente eran negativas.

Recall = 0.9602 → De todas las reseñas negativas reales, acertó el 96%.

F1 = 0.6855 → Balance entre precision y recall, relativamente aceptable.

Clase 1 (positiva)

Precision = 0.7997 → De las que predijo como positivas, el 80% eran realmente positivas.

Recall = 0.1587 → Pero solo identificó correctamente el 15% de todas las positivas reales (falló en la mayoría).

F1 = 0.2649 → Muy bajo, indica que la clase positiva casi no se está aprendiendo.

4) Totales
accuracy       0.5595
macro avg     0.6663    0.5595    0.4752
weighted avg  0.6663    0.5595    0.4752


Macro avg → promedio simple de métricas en ambas clases.

Weighted avg → promedio ponderado por la cantidad de ejemplos en cada clase.

Ambos coinciden porque las clases están balanceadas (12,500 negativas y 12,500 positivas).

5) Conclusión

El modelo aprendió a detectar muy bien la clase negativa (recall ~96%), pero casi ignora la clase positiva (recall ~16%).

Esto se llama sesgo de clase (class bias) → la red “juega seguro” prediciendo casi siempre 0 (negativo) porque así logra mayor accuracy global, pero no generaliza.

8) Probar con textos nuevos (rápido)

El dataset IMDB ya viene tokenizado como índices, por lo que no tiene diccionario de palabras accesible directamente.
Para probar textos crudos fácilmente, a continuación te dejo una Opción B con un minidataset en español usando Tokenizer.

In [ ]:
# Opción B (mini dataset en español) – Tokenizer + LSTM
# Útil para explicar el flujo texto → tokens → padding → LSTM con frases cortas en español.
# B1) Minidataset en español (toy)
# @title Minidataset en español (positivo/negativo)
textos = [
    "me encantó la película",               # 1
    "excelente actuación y banda sonora",   # 1
    "una obra maestra",                     # 1
    "totalmente recomendable",              # 1
    "buena historia y personajes",          # 1
    "no me gustó",                          # 0
    "muy aburrida y lenta",                 # 0
    "pésima dirección",                     # 0
    "mal guion y actuaciones",              # 0
    "no la volvería a ver",                 # 0
    "fue divertida y emocionante",          # 1
    "una experiencia terrible",             # 0,
    "me hizo reír mucho",                   # 1,
    "demasiado larga y cansina",            # 0,
    "simplemente espectacular",             # 1
]
labels = [1,1,1,1,1, 0,0,0,0,0, 1,0,1,0,1]  # 1=positivo, 0=negativo

textos = np.array(textos)
labels = np.array(labels)

In [ ]:
# B2) Tokenizer + secuencias + padding
# @title Tokenización y padding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

vocab_size = 1000
oov_token  = "<OOV>"     # token para palabras desconocidas
max_len    = 12          # longitud fija para padding (frases cortas)

tok = Tokenizer(num_words=vocab_size, oov_token=oov_token)
tok.fit_on_texts(textos)

seqs = tok.texts_to_sequences(textos)
X = pad_sequences(seqs, maxlen=max_len, padding="post", truncating="post")
y = labels

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=SEED, stratify=y)
X_train, X_val,  y_train, y_val  = train_test_split(X_train, y_train, test_size=0.30, random_state=SEED, stratify=y_train)

print("X_train:", X_train.shape, "X_val:", X_val.shape, "X_test:", X_test.shape)
print("Vocab size real:", len(tok.word_index))
print("Ejemplo tokens:", seqs[0], "->", textos[0])

1) Dimensiones de los conjuntos
X_train: (7, 12)
X_val  : (3, 12)
X_test : (5, 12)


X_train (7, 12) → Hay 7 frases de entrenamiento, cada una convertida en una secuencia de 12 tokens.

X_val (3, 12) → Hay 3 frases para validación, también con longitud fija de 12.

X_test (5, 12) → Hay 5 frases para prueba.

 El 12 viene del max_len=12 que definimos en el padding → todas las frases (largas o cortas) se ajustan a longitud fija.

Si la frase tiene menos de 12 palabras, se rellena con ceros (padding).

Si tiene más de 12, se recorta.

 2) Vocab size real
Vocab size real: 44


Significa que el Tokenizer encontró 44 palabras distintas en el dataset.

Ejemplo: "me", "gustó", "aburrida", "película", etc.

A cada palabra se le asigna un índice entero único.

 El modelo no trabaja con texto directamente, sino con números que representan palabras.

3) Ejemplo de tokens
Ejemplo tokens: [3, 7, 4, 8] -> me encantó la película


La frase "me encantó la película" fue convertida en una secuencia de enteros.

Según el diccionario del Tokenizer:

3 → "me"

7 → "encantó"

4 → "la"

8 → "película"

Estos números son los que luego se pasan a la capa Embedding, que transforma cada token en un vector denso con significado semántico.

Resumen:

Tokenizer convierte texto → enteros.

Padding hace que todas las frases tengan la misma longitud (aquí 12).

El dataset queda en forma de tensores:

X_train con 7 frases,

X_val con 3 frases,

X_test con 5 frases.

Ejemplo:

Texto original: "me encantó la película"

Tokens: [3, 7, 4, 8]

Con padding: [3, 7, 4, 8, 0, 0, 0, 0, 0, 0, 0, 0]

Así el modelo LSTM puede leer las frases como secuencias de números.

In [ ]:
# B3) Modelo LSTM (pequeño) y entrenamiento
# @title LSTM pequeño para minidataset español
embedding_dim = 32
lstm_units    = 32

model_es = keras.Sequential([
    layers.Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len),
    layers.SpatialDropout1D(0.2),
    layers.LSTM(lstm_units),
    layers.Dropout(0.3),
    layers.Dense(1, activation="sigmoid")
])
model_es.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model_es.summary()

early = keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)

hist_es = model_es.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=8,
    callbacks=[early],
    verbose=0
)

print("Mejor val_acc:", np.max(hist_es.history["val_accuracy"]))

1) La advertencia
UserWarning: Argument `input_length` is deprecated. Just remove it.


Antes se usaba input_length=max_len en la capa Embedding.

Hoy en día Keras ya no lo necesita porque deduce automáticamente la longitud a partir de los datos.
 No es un error, solo un aviso de que puedes quitar ese argumento.

 2) Resumen del modelo
Model: "sequential_1"
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
...
└─────────────────────────────────┴────────────────────────┴───────────────┘
 Total params: 0 (0.00 B)


Aquí aparecen "?" y 0 parámetros, porque:

El modelo aún no está "build" (no se construyó completamente).

Keras solo puede calcular los parámetros una vez que recibe datos reales (fit(), build()).

Después del primer entrenamiento, si corres model.summary() otra vez, ya mostrará los parámetros correctos.

 Esto pasa mucho con modelos que usan Embedding + LSTM, porque necesitan conocer input_dim (vocab_size) y max_len para calcular los tamaños.

 3) Capas del modelo

Embedding

Convierte tokens (números) en vectores densos de dimensión embedding_dim=32.

Parámetros = vocab_size × embedding_dim.

Ejemplo: si vocab_size=1000 y embedding_dim=32 → 32,000 parámetros.

SpatialDropout1D

Apaga aleatoriamente vectores de palabras completos durante el entrenamiento.

Previene sobreajuste en secuencias cortas.

LSTM (32 unidades)

Procesa la frase palabra por palabra, reteniendo memoria de contexto.

Número de parámetros = fórmula de LSTM:

4 × (units × (units + input_dim + 1))


Para 32 unidades, suele dar miles de parámetros.

Dropout

Apaga conexiones en la salida de la LSTM para robustez.

Dense (sigmoid)

1 neurona → salida entre 0 y 1.

Decide positivo (1) o negativo (0).

 4) Resultado de validación
Mejor val_acc: 0.3333333432674408


La mejor accuracy en validación fue ~33%.

Como tu validación tiene 3 frases nada más (X_val = (3, 12)), esa métrica puede fluctuar mucho.

Ejemplo: si acertó 1 de 3 → 33%.

Si acertó 2 de 3 → 66%.

 Esto pasa porque el minidataset es muy pequeño (solo 15 frases en total).
No es que el modelo sea malo, sino que no hay suficientes datos para entrenar de forma estable.

En resumen para tus estudiantes

El warning es solo porque input_length ya no se necesita.

El resumen muestra 0 parámetros porque el modelo aún no procesó datos; después de fit() sí aparecerán.

Las capas hacen: texto → embedding → LSTM → probabilidad.

El val_acc = 33% refleja que con muy pocos ejemplos la métrica es inestable.

In [ ]:
# B4) Evaluación + reporte
# @title Evaluación en test (minidataset español)
test_loss, test_acc = model_es.evaluate(X_test, y_test, verbose=0)
print(f"Accuracy en test (ES): {test_acc:.3f} | Pérdida: {test_loss:.3f}")

y_prob = model_es.predict(X_test, verbose=0).reshape(-1)
y_pred = (y_prob >= 0.5).astype(int)
print("Matriz de confusión:\n", confusion_matrix(y_test, y_pred))
print("\nReporte de clasificación:\n", classification_report(y_test, y_pred, digits=3))

1) Accuracy y pérdida
Accuracy en test (ES): 0.600 | Pérdida: 0.689


El modelo acertó el 60% de los casos en el test.

La pérdida (0.689) es alta (cercana a 0.693, que sería azar puro en clasificación binaria).

 Conclusión: el modelo aprende algo, pero aún se comporta muy cerca al azar (probablemente por el dataset tan pequeño).

 2) Matriz de confusión
 [[0 2]
  [0 3]]


Filas = clases reales.

Columnas = clases predichas.

Interpretación:

Clase 0 (negativo):

Había 2 ejemplos → el modelo predijo ambos como 1 (ninguno como 0).

Clase 1 (positivo):

Había 3 ejemplos → el modelo predijo los 3 correctamente como 1.

 El modelo clasificó todo como positivo (1).

 3) Reporte de clasificación
precision    recall  f1-score   support

Clase 0 (negativo)

Precision = 0.000 → nunca predijo “0”, por lo tanto no hay muestras correctas de clase 0.

Recall = 0.000 → de las 2 que eran negativas, no acertó ninguna.

F1 = 0.000 → el modelo ignora totalmente la clase 0.

Clase 1 (positivo)

Precision = 0.600 → de las 5 predicciones positivas que hizo, 3 eran correctas (60%).

Recall = 1.000 → de todas las positivas reales (3), acertó las 3.

F1 = 0.750 → balance entre precision y recall, relativamente bueno.

Totales

Accuracy = 0.600 (60%)

Macro avg = promedio simple de métricas (0 y 1).

Weighted avg = promedio ponderado (con más peso a la clase con más ejemplos).

 4) Advertencia (UndefinedMetricWarning)
UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples


Ocurre porque el modelo nunca predijo la clase 0.

Entonces la precision de la clase 0 no se puede calcular (división por cero), y sklearn la reemplaza con 0.

 5) Resumen para tus estudiantes

El modelo logró 60% de aciertos, pero en realidad solo aprendió a decir “positivo” (clase 1).

Esto pasa porque el dataset es muy pequeño y desbalanceado (solo 5 ejemplos en test).

Por eso, en la matriz de confusión vemos que ignora completamente la clase negativa (0).

El warning aparece porque sklearn no puede calcular la precision cuando no hay predicciones de una clase.

 En conclusión: este ejercicio sirve para mostrar el flujo completo texto → tokens → padding → LSTM, pero no para medir rendimiento real, porque el dataset es demasiado chico.

In [ ]:
# B5) Predicciones con frases nuevas (en español)
# @title Probar con frases nuevas (español)
def predecir_frase(frase):
    seq = tok.texts_to_sequences([frase])
    pad = pad_sequences(seq, maxlen=max_len, padding="post", truncating="post")
    prob = model_es.predict(pad, verbose=0)[0,0]
    clase = 1 if prob >= 0.5 else 0
    print(f"Frase: {frase}")
    print(f"Prob(positiva) = {prob:.3f} -> Clase predicha: {clase} ({'positiva' if clase==1 else 'negativa'})\n")

ejemplos = [
    "me encantó y la volvería a ver",
    "fue aburrida y demasiado larga",
    "actuaciones brillantes pero el guion regular",
    "horrible experiencia, no la recomiendo",
]
for t in ejemplos:
    predecir_frase(t)

1) Cómo funciona la predicción

Cada frase se tokeniza → se convierte en números (tokens).

Se aplica padding → todas las frases tienen la misma longitud (12 en tu caso).

El modelo pasa esos tokens por:

Embedding (palabras → vectores densos),

LSTM (lee la secuencia),

Dense con sigmoide (da una probabilidad entre 0 y 1).

 El resultado es un número entre 0 y 1 = Probabilidad de ser positivo.

Si prob >= 0.5 → se predice clase 1 (positiva).

Si prob < 0.5 → se predice clase 0 (negativa).

2) Resultados que obtuviste
Frase: me encantó y la volvería a ver
Prob(positiva) = 0.512 -> Clase predicha: 1 (positiva)

Frase: fue aburrida y demasiado larga
Prob(positiva) = 0.512 -> Clase predicha: 1 (positiva)

Frase: actuaciones brillantes pero el guion regular
Prob(positiva) = 0.512 -> Clase predicha: 1 (positiva)

Frase: horrible experiencia, no la recomiendo
Prob(positiva) = 0.513 -> Clase predicha: 1 (positiva)


Todas las frases, incluso las claramente negativas, fueron clasificadas como positivas.

El modelo siempre da valores alrededor de 0.51–0.513, apenas por encima del umbral 0.5.

3) ¿Qué significa esto?

El modelo no aprendió realmente el sentimiento → está respondiendo casi siempre lo mismo.

Esto pasa porque:

El dataset era muy pequeño (solo 15 frases).

En el entrenamiento, el modelo terminó sesgado hacia la clase positiva (como vimos en la evaluación).

Los embeddings y la LSTM no tuvieron suficiente datos para distinguir bien entre positivo y negativo.

En términos simples: el modelo aprendió a decir “positivo” por default, porque con tan pocos ejemplos esa estrategia ya le da un accuracy aceptable en train.

4) Enseñanza para tus estudiantes

Con datasets pequeños, los modelos de deep learning no generalizan bien.

El modelo debe entrenarse con miles de ejemplos balanceados para que aprenda patrones reales (palabras como “horrible” → negativo, “encantó” → positivo).

Aquí sirve como ejemplo pedagógico para mostrar el flujo completo (texto → tokens → padding → LSTM → predicción), pero no como un clasificador real.

En resumen:
El modelo está devolviendo siempre casi el mismo valor (~0.51) y prediciendo todo como positivo. Esto refleja que no aprendió a diferenciar sentimientos por falta de datos y entrenamiento.